In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, json, time, gc, random, math, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
import transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class PairwiseDataset(Dataset):
    def __init__(self, paths, tokenizer, max_length=512):
        self.data = []
        for p in paths:
            with open(p) as f:
                for line in f:
                    if line.strip(): self.data.append(json.loads(line))
        self.tok = tokenizer; self.max_len = max_length
    def encode(self, q, p):
        return self.tok(q, p, max_length=self.max_len, padding="max_length",
                        truncation=True, return_tensors="pt")
    def __getitem__(self, idx):
        d = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {"pos_input_ids": pos["input_ids"].squeeze(),
                "pos_attention_mask": pos["attention_mask"].squeeze(),
                "neg_input_ids": neg["input_ids"].squeeze(),
                "neg_attention_mask": neg["attention_mask"].squeeze()}
    def __len__(self): return len(self.data)

class ListwiseRankDataset(Dataset):
    def __init__(self, rerank_path, tokenizer, max_length=512, max_candidates=20):
        self.tok = tokenizer; self.max_len = max_length; self.max_cand = max_candidates
        self.records = []
        with open(rerank_path) as f:
            for line in f:
                if line.strip(): self.records.append(json.loads(line))
        print(f"Total records: {len(self.records)}")
    def __getitem__(self, idx):
        d = self.records[idx]; query = d["question"]
        candidates = d["candidates"][:self.max_cand]
        encodings, ranks, bge_scores = [], [], []
        for c in candidates:
            enc = self.tok(query, c["chunk"], max_length=self.max_len,
                           padding="max_length", truncation=True, return_tensors="pt")
            encodings.append({"input_ids": enc["input_ids"].squeeze(),
                              "attention_mask": enc["attention_mask"].squeeze()})
            ranks.append(c.get("rank", 0)); bge_scores.append(c.get("bge_score", 0.0))
        return {"encodings": encodings,
                "ranks": torch.tensor(ranks, dtype=torch.float),
                "bge_scores": torch.tensor(bge_scores, dtype=torch.float)}
    def __len__(self): return len(self.records)

def collate_listwise_rank(batch):
    all_ids, all_masks, all_ranks, all_scores, sizes = [], [], [], [], []
    for item in batch:
        sizes.append(len(item["encodings"]))
        for enc in item["encodings"]:
            all_ids.append(enc["input_ids"]); all_masks.append(enc["attention_mask"])
        all_ranks.append(item["ranks"]); all_scores.append(item["bge_scores"])
    return {"input_ids": torch.stack(all_ids), "attention_mask": torch.stack(all_masks),
            "ranks": all_ranks, "bge_scores": all_scores, "sizes": sizes}

class ADRMSELoss(nn.Module):
    def __init__(self): super().__init__(); self.mse = nn.MSELoss()
    def forward(self, student_logits, teacher_ranks):
        n = teacher_ranks.size(-1)
        rank_signal = 1.0 - 2.0 * teacher_ranks / (n - 1)
        s_mean = student_logits.mean(dim=-1, keepdim=True)
        s_std = student_logits.std(dim=-1, keepdim=True) + 1e-8
        return self.mse((student_logits - s_mean) / s_std, rank_signal)

class RankNetSoftLoss(nn.Module):
    def forward(self, student_logits, teacher_scores):
        n = student_logits.size(-1)
        s_diff = student_logits.unsqueeze(2) - student_logits.unsqueeze(1)
        t_diff = teacher_scores.unsqueeze(2) - teacher_scores.unsqueeze(1)
        mask = ~torch.eye(n, dtype=torch.bool, device=student_logits.device)
        return F.binary_cross_entropy(torch.sigmoid(s_diff)[:, mask],
                                      torch.sigmoid(t_diff)[:, mask])

class StageALoss(nn.Module):
    def __init__(self): super().__init__(); self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pos_logits, neg_logits):
        return (self.bce(pos_logits, torch.ones_like(pos_logits)) +
                self.bce(neg_logits, torch.zeros_like(neg_logits))) / 2

class ListwiseKLLoss(nn.Module):
    def __init__(self, temperature=2.0): super().__init__(); self.T = temperature
    def forward(self, student_logits, teacher_scores):
        s = F.log_softmax(student_logits / self.T, dim=-1)
        t = F.softmax(teacher_scores / self.T, dim=-1)
        return F.kl_div(s, t, reduction="batchmean") * (self.T ** 2)

def evaluate_pairwise(model, dev_loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for batch in dev_loader:
            pos = model(input_ids=batch["pos_input_ids"].to(device),
                        attention_mask=batch["pos_attention_mask"].to(device)).logits
            neg = model(input_ids=batch["neg_input_ids"].to(device),
                        attention_mask=batch["neg_attention_mask"].to(device)).logits
            correct += (pos > neg).sum().item(); total += pos.size(0)
    return correct / total

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | Helpers loaded ✓")

PyTorch 2.11.0+cu128 | CUDA True | Helpers loaded ✓


In [4]:
from pathlib import Path
import os, json, torch

BASE = Path("/content/drive/MyDrive/Data/archive")

MINILM_BASE = str(BASE / "ms-marco-MiniLM-L12-v2")
MMARCO_BASE = str(BASE / "mmarco-mMiniLMv2-L12-H384-v1")
BGE_DIR     = str(BASE / "bge-reranker-v2-m3")
PHORANKER   = str(BASE / "PhoRanker")
VIRANKER    = str(BASE / "ViRanker")

DOMAIN_TRAIN = str(BASE / "/content/drive/MyDrive/Data/domain_train_final_train_no_synth.jsonl")
DOMAIN_DEV   = str(BASE / "domain_train_final_dev.jsonl")
RERANK_991   = str(BASE / "retrieve_rerank_991.jsonl")
MMARCO_DATA  = str("/content/drive/MyDrive/Data/mmarco_vi_50k.jsonl")

TEST_Q        = str("/content/drive/MyDrive/Data/question.json")
CHUNK_DIR     = BASE / "chunk_outputs1_finals"
EMB_FT        = str("outputs/embed_clean_baseline/checkpoints/embed_clean_mnr_1stage_seed42")
MARGIN_TRAIN  = str(BASE / "domain_train_with_teacher_scores.jsonl")

ABL = "/content/drive/MyDrive/Data/archive/ablation"
os.makedirs(ABL, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

with open(TEST_Q, encoding="utf-8") as f:
    test_raw = json.load(f)

test_queries = [
    {"question": q["question"], "gold": set(q.get("gold_chunk_ids", []))}
    for q in test_raw
    if q.get("gold_chunk_ids")
]

corpus = {}
for doc_dir in CHUNK_DIR.iterdir():
    if not doc_dir.is_dir():
        continue
    doc_scope = doc_dir.name
    for cf in doc_dir.glob("*.json"):
        with open(cf, encoding="utf-8") as f:
            chunk_records = json.load(f)
        for i, rec in enumerate(chunk_records):
            text = str(rec.get("page_content", "")).strip()
            md = rec.get("metadata", {}) or {}
            raw = str(md.get("chunk_id") or f"chunk::{md.get('chunk_index', i)}").strip()
            if not raw or not text:
                continue
            cid = raw if raw.startswith(f"{doc_scope}::") else f"{doc_scope}::{raw}"
            corpus[cid] = text

cids, texts = list(corpus.keys()), list(corpus.values())

print(f"Device: {device}")
print(f"Test queries: {len(test_queries)} | Full corpus: {len(corpus)} chunks")

Device: cuda
Test queries: 390 | Full corpus: 813 chunks


In [4]:
def train_stage_a(domain_train_path, mmarco_path, dev_path, output_dir,
                  base_model=None, epochs=5, batch_size=32, lr=2e-5,
                  max_length=512, domain_upsample=8, patience=2, seed=42):
    if base_model is None: base_model = MINILM_BASE
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model, num_labels=1, ignore_mismatched_sizes=True).to(device)
    print(f"Base: {base_model.split('/')[-1]} | Seed: {seed}")

    domain_paths = [domain_train_path] * domain_upsample
    all_paths = domain_paths + ([mmarco_path] if mmarco_path else [])
    train_ds = PairwiseDataset(all_paths, tokenizer, max_length)
    dev_ds   = PairwiseDataset([dev_path], tokenizer, max_length)
    print(f"Train: {len(train_ds):,} | Dev: {len(dev_ds):,}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, worker_init_fn=lambda w: set_seed(seed+w))
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, num_workers=0)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer,
                  int(0.1*total_steps), total_steps)
    criterion = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    best_acc = 0; no_improve = 0

    for epoch in range(epochs):
        model.train(); total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos = model(input_ids=batch["pos_input_ids"].to(device),
                        attention_mask=batch["pos_attention_mask"].to(device)).logits
            neg = model(input_ids=batch["neg_input_ids"].to(device),
                        attention_mask=batch["neg_attention_mask"].to(device)).logits
            loss = criterion(pos, neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); total_loss += loss.item()
        acc = evaluate_pairwise(model, dev_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Dev Acc: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc; no_improve = 0
            model.save_pretrained(f"{output_dir}/best"); tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}"); break
    print(f"Stage A done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


def train_stage_b_ranknet(stage_a_checkpoint, rerank_path, domain_train_path, dev_path,
                          output_dir, loss_type="adr_mse", epochs=5, batch_size=8,
                          lr=1e-5, max_length=512, alpha=0.7, patience=2, seed=42):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(stage_a_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(stage_a_checkpoint).to(device)
    print(f"Loss: {loss_type} | alpha={alpha} | Seed: {seed}")

    kd_ds = ListwiseRankDataset(rerank_path, tok, max_length)
    kd_loader = DataLoader(kd_ds, batch_size=batch_size, shuffle=True,
                           collate_fn=collate_listwise_rank, num_workers=0,
                           worker_init_fn=lambda w: set_seed(seed+w))
    cl_ds = PairwiseDataset([domain_train_path], tok, max_length)
    cl_loader = DataLoader(cl_ds, batch_size=batch_size*2, shuffle=True, num_workers=0)
    cl_iter = iter(cl_loader)
    dev_loader = DataLoader(PairwiseDataset([dev_path], tok, max_length), batch_size=32, num_workers=0)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(kd_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)

    if   loss_type == "adr_mse":  kd_crit = ADRMSELoss()
    elif loss_type == "ranknet":  kd_crit = RankNetSoftLoss()
    elif loss_type == "kl":       kd_crit = ListwiseKLLoss()
    # elif loss_type == "discount": kd_crit = DiscountedADRMSELoss()
    # elif loss_type == "topk":     kd_crit = TopKADRMSELoss(k=10)
    cl_crit = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    best_acc = 0; no_improve = 0
    uses_scores = loss_type in ("ranknet", "kl")

    for epoch in range(epochs):
        model.train(); total_loss = total_kd = total_cl = 0
        for batch in kd_loader:
            optimizer.zero_grad()
            all_logits = model(input_ids=batch["input_ids"].to(device),
                               attention_mask=batch["attention_mask"].to(device)).logits.squeeze(-1)
            kd_loss = torch.tensor(0.0, device=device); offset = 0
            for i, size in enumerate(batch["sizes"]):
                q_logits = all_logits[offset:offset+size].unsqueeze(0)
                if uses_scores:
                    q_scores = batch["bge_scores"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_scores)
                else:
                    q_ranks = batch["ranks"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_ranks)
                offset += size
            kd_loss /= len(batch["sizes"])
            try: cl_batch = next(cl_iter)
            except StopIteration: cl_iter = iter(cl_loader); cl_batch = next(cl_iter)
            pos = model(input_ids=cl_batch["pos_input_ids"].to(device),
                        attention_mask=cl_batch["pos_attention_mask"].to(device)).logits
            neg = model(input_ids=cl_batch["neg_input_ids"].to(device),
                        attention_mask=cl_batch["neg_attention_mask"].to(device)).logits
            cl_loss = cl_crit(pos, neg)
            loss = alpha * kd_loss + (1 - alpha) * cl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item(); total_kd += kd_loss.item(); total_cl += cl_loss.item()
        acc = evaluate_pairwise(model, dev_loader, device)
        n = len(kd_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | "
              f"KD({loss_type}): {total_kd/n:.4f} | CL: {total_cl/n:.4f} | Dev Acc: {acc:.4f}")
        if acc > best_acc:
            best_acc = acc; no_improve = 0
            model.save_pretrained(f"{output_dir}/best"); tok.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}"); break
    print(f"{loss_type} done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"
print("Train functions loaded ✓")

Train functions loaded ✓


In [ ]:
class MarginMSELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pos_logits, neg_logits, teacher_margin):
        pos_logits = pos_logits.view(-1)
        neg_logits = neg_logits.view(-1)
        teacher_margin = teacher_margin.view(-1)
        return self.mse(pos_logits - neg_logits, teacher_margin)


class MarginMSEDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512, drop_negative_margin=False):
        self.data = []
        self.tok = tokenizer
        self.max_len = max_length

        bad = 0
        dropped = 0

        with open(path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                d = json.loads(line)

                q = str(d.get("query", "")).strip()
                pos = str(d.get("positive", "")).strip()
                neg = str(d.get("negative", "")).strip()

                if not q or not pos or not neg:
                    bad += 1
                    continue

                teacher_margin = d.get("teacher_margin", None)

                if teacher_margin is None:
                    if "pos_score" in d and "neg_score" in d:
                        teacher_margin = float(d["pos_score"]) - float(d["neg_score"])
                    elif "teacher_pos_score" in d and "teacher_neg_score" in d:
                        teacher_margin = float(d["teacher_pos_score"]) - float(d["teacher_neg_score"])
                    elif "bge_score_pos" in d and "bge_score_neg" in d:
                        teacher_margin = float(d["bge_score_pos"]) - float(d["bge_score_neg"])
                    else:
                        bad += 1
                        continue

                teacher_margin = float(teacher_margin)

                if drop_negative_margin and teacher_margin <= 0:
                    dropped += 1
                    continue

                self.data.append({
                    "query": q,
                    "positive": pos,
                    "negative": neg,
                    "teacher_margin": teacher_margin,
                })

        margins = [x["teacher_margin"] for x in self.data]
        print(f"MarginMSE data: {len(self.data):,} | bad={bad:,} | dropped={dropped:,}")
        if margins:
            print(
                f"teacher_margin min/mean/max = "
                f"{min(margins):.4f} / {sum(margins)/len(margins):.4f} / {max(margins):.4f}"
            )

    def encode(self, q, p):
        return self.tok(
            q,
            p,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        d = self.data[idx]

        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])

        return {
            "pos_input_ids": pos["input_ids"].squeeze(0),
            "pos_attention_mask": pos["attention_mask"].squeeze(0),
            "neg_input_ids": neg["input_ids"].squeeze(0),
            "neg_attention_mask": neg["attention_mask"].squeeze(0),
            "teacher_margin": torch.tensor(d["teacher_margin"], dtype=torch.float),
        }

    def __len__(self):
        return len(self.data)


def train_stage_b_margin_mse(
    base_checkpoint,
    train_path,
    dev_path,
    output_dir,
    domain_train_path=None,
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
    drop_negative_margin=False,
):
    set_seed(seed)

    tok = AutoTokenizer.from_pretrained(base_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(base_checkpoint).to(device)

    print(f"Loss: margin_mse | alpha={alpha} | Seed: {seed}")
    print("Base:", base_checkpoint)
    print("Train:", train_path)

    margin_ds = MarginMSEDataset(
        train_path,
        tok,
        max_length=max_length,
        drop_negative_margin=drop_negative_margin,
    )

    margin_loader = DataLoader(
        margin_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        worker_init_fn=lambda w: set_seed(seed + w),
    )

    cl_loader = None
    cl_iter = None
    cl_crit = StageALoss()

    if domain_train_path is not None and alpha < 1.0:
        cl_ds = PairwiseDataset([domain_train_path], tok, max_length)
        cl_loader = DataLoader(
            cl_ds,
            batch_size=batch_size * 2,
            shuffle=True,
            num_workers=0,
            worker_init_fn=lambda w: set_seed(seed + 1000 + w),
        )
        cl_iter = iter(cl_loader)
        print(f"CL data: {len(cl_ds):,}")

    dev_loader = DataLoader(
        PairwiseDataset([dev_path], tok, max_length),
        batch_size=32,
        num_workers=0,
    )

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(margin_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(0.05 * total_steps),
        total_steps,
    )

    margin_crit = MarginMSELoss()

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc = 0.0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_margin = 0.0
        total_cl = 0.0

        for batch in margin_loader:
            optimizer.zero_grad()

            pos_logits = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits.squeeze(-1)

            neg_logits = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits.squeeze(-1)

            teacher_margin = batch["teacher_margin"].to(device)

            margin_loss = margin_crit(pos_logits, neg_logits, teacher_margin)

            if cl_loader is not None:
                try:
                    cl_batch = next(cl_iter)
                except StopIteration:
                    cl_iter = iter(cl_loader)
                    cl_batch = next(cl_iter)

                cl_pos = model(
                    input_ids=cl_batch["pos_input_ids"].to(device),
                    attention_mask=cl_batch["pos_attention_mask"].to(device),
                ).logits

                cl_neg = model(
                    input_ids=cl_batch["neg_input_ids"].to(device),
                    attention_mask=cl_batch["neg_attention_mask"].to(device),
                ).logits

                cl_loss = cl_crit(cl_pos, cl_neg)
                loss = alpha * margin_loss + (1.0 - alpha) * cl_loss
            else:
                cl_loss = torch.tensor(0.0, device=device)
                loss = margin_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            total_margin += margin_loss.item()
            total_cl += cl_loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n = len(margin_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {total_loss / n:.4f} | "
            f"MarginMSE: {total_margin / n:.4f} | "
            f"CL: {total_cl / n:.4f} | "
            f"Dev Acc: {acc:.4f}"
        )

        if acc > best_acc:
            best_acc = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tok.save_pretrained(f"{output_dir}/best")
            print(f"  -> Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    print(f"margin_mse done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Margin-MSE train function loaded")

Margin-MSE train function loaded


In [ ]:
ckpt_stage_a = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=None,
    dev_path=DOMAIN_DEV,
    output_dir=f"{ABL}/stage_a_minilm_colab_no_synth",
    base_model=MINILM_BASE,
    domain_upsample=1,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    patience=2,
    seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base: ms-marco-MiniLM-L12-v2 | Seed: 42
Train: 2,328 | Dev: 305
Epoch 1/5 | Loss: 0.7096 | Dev Acc: 0.7410


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7410)
Epoch 2/5 | Loss: 0.4362 | Dev Acc: 0.7705


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7705)
Epoch 3/5 | Loss: 0.2890 | Dev Acc: 0.7869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7869)
Epoch 4/5 | Loss: 0.2098 | Dev Acc: 0.7869
Epoch 5/5 | Loss: 0.1731 | Dev Acc: 0.7902


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7902)
Stage A done. Best dev acc: 0.7902


In [ ]:
STAGE_A_CKPT = "/content/drive/MyDrive/Data/archive/ablation/stage_a_minilm_colab_no_synth/best"

RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"
DOMAIN_TRAIN_STAGE_B = DOMAIN_TRAIN
DOMAIN_DEV_STAGE_B = DOMAIN_DEV

ABL_LOSS = f"{ABL}/loss_ablation_no_synth_stage_a"

LOSS_CFG = {
    "epochs": 5,
    "batch_size": 8,
    "lr": 1e-5,
    "alpha": 0.7,
    "patience": 2,
    "seed": 42,
}

ckpt_adrmse = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS}/stage_b_adrmse",
    loss_type="adr_mse",
    **LOSS_CFG,
)

ckpt_kl = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS}/stage_b_kl",
    loss_type="kl",
    **LOSS_CFG,
)

ckpt_ranknet = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS}/stage_b_ranknet",
    loss_type="ranknet",
    **LOSS_CFG,
)

ckpt_margin = train_stage_b_margin_mse(
    base_checkpoint=STAGE_A_CKPT,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS}/stage_b_margin",
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    epochs=5,
    batch_size=8,
    lr=1e-5,
    alpha=0.7,
    patience=2,
    seed=42,
)

print("DONE")
print("ADR-MSE :", ckpt_adrmse)
print("KL      :", ckpt_kl)
print("RankNet :", ckpt_ranknet)
print("Margin  :", ckpt_margin)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.5823 | KD(adr_mse): 0.7535 | CL: 0.1828 | Dev Acc: 0.8262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8262)
Epoch 2/5 | Loss: 0.5275 | KD(adr_mse): 0.6805 | CL: 0.1705 | Dev Acc: 0.8426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  → Saved best (acc=0.8459)
Epoch 5/5 | Loss: 0.4657 | KD(ranknet): 0.6032 | CL: 0.1448 | Dev Acc: 0.8459
ranknet done. Best dev acc: 0.8459


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: margin_mse | alpha=0.7 | Seed: 42
Base: /content/drive/MyDrive/Data/archive/ablation/stage_a_minilm_colab_no_synth/best
Train: /content/drive/MyDrive/Data/archive/domain_train_with_teacher_scores.jsonl
MarginMSE data: 2,689 | bad=0 | dropped=0
teacher_margin min/mean/max = -7.5323 / 4.1303 / 15.7282
CL data: 2,328
Epoch 1/5 | Loss: 4.1512 | MarginMSE: 5.8162 | CL: 0.2663 | Dev Acc: 0.8098


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> Saved best (acc=0.8098)
Epoch 2/5 | Loss: 2.4456 | MarginMSE: 3.3849 | CL: 0.2539 | Dev Acc: 0.8000
Epoch 3/5 | Loss: 2.0911 | MarginMSE: 2.8820 | CL: 0.2455 | Dev Acc: 0.8066
Early stopping at epoch 3
margin_mse done. Best dev acc: 0.8098
DONE
ADR-MSE : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_adrmse/best
KL      : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_kl/best
RankNet : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_ranknet/best
Margin  : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_margin/best


In [5]:
import json, math, gc
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ===== Config =====
# EMB_FT = "/content/drive/MyDrive/Data/archive/outputs/embed_clean_baseline/checkpoints/embed_clean_mnr_1stage_seed42"
EMB_FT = "/content/drive/MyDrive/Data/archive/outputs/seed_sweep/baseline_seed42"
TOP_K_DENSE = 20
EMB_DIM = 512

print(f"Queries: {len(test_queries)} | Corpus chunks: {len(cids)}")
print("Embedding retriever:", EMB_FT)
print("Embedding dim used:", EMB_DIM)


def encode_truncated(model, inputs, dim=512, batch_size=64, show_progress_bar=True):
    vec = model.encode(
        inputs,
        batch_size=batch_size,
        normalize_embeddings=False,   # cắt trước, normalize sau
        convert_to_numpy=True,
        show_progress_bar=show_progress_bar,
    )

    if vec.shape[1] < dim:
        raise ValueError(f"Model output dim = {vec.shape[1]}, nhỏ hơn dim cần dùng = {dim}")

    vec = vec[:, :dim]
    vec = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-12)
    return vec.astype(np.float32)


def _dcg_at_k(rels, k):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels[:k]))


def _metrics_one(ranked_ids, gold_set, k_recall=5, k_mrr=10, k_ndcg=10):
    gold_set = set(gold_set)

    hit1 = float(len(ranked_ids) > 0 and ranked_ids[0] in gold_set)
    recall5 = float(any(cid in gold_set for cid in ranked_ids[:k_recall]))

    mrr10 = 0.0
    for rank, cid in enumerate(ranked_ids[:k_mrr], start=1):
        if cid in gold_set:
            mrr10 = 1.0 / rank
            break

    rels = [1.0 if cid in gold_set else 0.0 for cid in ranked_ids[:k_ndcg]]
    dcg = _dcg_at_k(rels, k_ndcg)

    ideal_n = min(len(gold_set), k_ndcg)
    idcg = _dcg_at_k([1.0] * ideal_n + [0.0] * (k_ndcg - ideal_n), k_ndcg)
    ndcg10 = dcg / idcg if idcg > 0 else 0.0

    return hit1, recall5, mrr10, ndcg10


def summarize_metrics(metrics, name, n):
    return {
        "name": name,
        "n": n,
        "Hit@1": float(np.mean([m[0] for m in metrics])),
        "Recall@5": float(np.mean([m[1] for m in metrics])),
        "MRR@10": float(np.mean([m[2] for m in metrics])),
        "NDCG@10": float(np.mean([m[3] for m in metrics])),
    }


def print_result(result):
    print("\n" + "=" * 80)
    print(result["name"])
    print("=" * 80)
    print(f"n:         {result['n']}")
    print(f"Hit@1:    {result['Hit@1']:.4f}")
    print(f"Recall@5: {result['Recall@5']:.4f}")
    print(f"MRR@10:   {result['MRR@10']:.4f}")
    print(f"NDCG@10:  {result['NDCG@10']:.4f}")


# 1. Dense retrieve top-20 bằng embedding 512d
def build_dense_topk_pool_512(embed_model_path=EMB_FT, top_k=20, dim=512, batch_size=64):
    emb = SentenceTransformer(embed_model_path, device=str(device))
    emb.max_seq_length = 1024

    # Check dim
    probe = emb.encode(["test"], normalize_embeddings=False, convert_to_numpy=True)
    print("Raw embedding shape:", probe.shape)

    print("Encoding corpus...")
    corpus_emb = encode_truncated(
        emb,
        texts,
        dim=dim,
        batch_size=batch_size,
        show_progress_bar=True,
    )

    queries = [q["question"] for q in test_queries]

    print("Encoding queries...")
    query_emb = encode_truncated(
        emb,
        queries,
        dim=dim,
        batch_size=batch_size,
        show_progress_bar=True,
    )

    pool = []
    pre_ranked = []

    print("Building dense top-k pool...")
    for qi, q in enumerate(tqdm(queries)):
        sims = corpus_emb @ query_emb[qi]
        top_idx = np.argsort(-sims)[:top_k]

        cand_ids = [cids[i] for i in top_idx]
        cand_texts = [texts[i] for i in top_idx]
        cand_scores = [float(sims[i]) for i in top_idx]

        pool.append({
            "question": q,
            "gold": list(test_queries[qi]["gold"]),
            "candidate_ids": cand_ids,
            "candidate_texts": cand_texts,
            "dense_scores": cand_scores,
        })
        pre_ranked.append(cand_ids)

    pre_metrics = [
        _metrics_one(ranked, test_queries[i]["gold"])
        for i, ranked in enumerate(pre_ranked)
    ]

    pre_result = summarize_metrics(
        pre_metrics,
        name=f"Dense retriever top-{top_k} ({dim}d) before rerank",
        n=len(pre_metrics),
    )
    print_result(pre_result)

    del emb, corpus_emb, query_emb
    gc.collect()
    torch.cuda.empty_cache()

    return pool, pre_result


@torch.no_grad()
def _score_reranker_pairs(model, tokenizer, query, docs, batch_size=32, max_length=512):
    model.eval()
    scores = []

    for i in range(0, len(docs), batch_size):
        batch_docs = docs[i:i + batch_size]
        enc = tokenizer(
            [query] * len(batch_docs),
            batch_docs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(model.device)

        logits = model(**enc).logits
        if logits.ndim == 2 and logits.size(-1) == 1:
            logits = logits.squeeze(-1)
        elif logits.ndim == 2:
            logits = logits[:, 0]

        scores.extend(logits.detach().float().cpu().tolist())

    return np.asarray(scores, dtype=np.float32)


# 2. Rerank dense top-20
def benchmark_dense_then_rerank(
    reranker_path,
    name,
    dense_pool,
    batch_size=32,
    max_length=512,
    save_rank_path=None,
):
    tokenizer = AutoTokenizer.from_pretrained(reranker_path, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(reranker_path).to(device)

    metrics = []
    rows = []

    for qi, rec in enumerate(tqdm(dense_pool, desc=name)):
        q = rec["question"]
        gold = set(rec["gold"])
        cand_ids = rec["candidate_ids"]
        cand_texts = rec["candidate_texts"]

        scores = _score_reranker_pairs(
            model=model,
            tokenizer=tokenizer,
            query=q,
            docs=cand_texts,
            batch_size=batch_size,
            max_length=max_length,
        )

        order = np.argsort(-scores)
        ranked_ids = [cand_ids[i] for i in order]

        metric = _metrics_one(ranked_ids, gold)
        metrics.append(metric)

        rows.append({
            "idx": qi,
            "question": q,
            "gold": list(gold),
            "top1": ranked_ids[0] if ranked_ids else None,
            "hit1": metric[0],
            "recall5": metric[1],
            "mrr10": metric[2],
            "ndcg10": metric[3],
            "top10": ranked_ids[:10],
            "top10_rerank_scores": [float(scores[i]) for i in order[:10]],
        })

    result = summarize_metrics(metrics, name=name, n=len(dense_pool))
    print_result(result)

    if save_rank_path:
        with open(save_rank_path, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("Saved:", save_rank_path)

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result, rows

Queries: 390 | Corpus chunks: 813
Embedding retriever: /content/drive/MyDrive/Data/archive/outputs/seed_sweep/baseline_seed42
Embedding dim used: 512


In [6]:
dense_pool, res_dense = build_dense_topk_pool_512(
    embed_model_path=EMB_FT,
    top_k=TOP_K_DENSE,
    dim=EMB_DIM,
    batch_size=64,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Raw embedding shape: (1, 1024)
Encoding corpus...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Building dense top-k pool...


  0%|          | 0/390 [00:00<?, ?it/s]


Dense retriever top-20 (512d) before rerank
n:         390
Hit@1:    0.7128
Recall@5: 0.9154
MRR@10:   0.7961
NDCG@10:  0.8328


In [ ]:
# ===== Benchmark 4 Stage-B loss models =====

LOSS_MODELS = {
    "ADR-MSE": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_adrmse/best",
    "ListwiseKL": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_kl/best",
    "RankNet": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_ranknet/best",
    "Margin-MSE": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_margin/best",
}

RANK_OUT_DIR = "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_outputs"
Path(RANK_OUT_DIR).mkdir(parents=True, exist_ok=True)

# Build dense pool 1 lần nếu chưa có
if "dense_pool" not in globals():
    dense_pool, dense_pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )

loss_results = {}
loss_rows = {}

for name, ckpt in LOSS_MODELS.items():
    print("\n" + "#" * 90)
    print(name)
    print(ckpt)
    print("#" * 90)

    result, rows = benchmark_dense_then_rerank(
        reranker_path=ckpt,
        name=name,
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=f"{RANK_OUT_DIR}/rank_{name.lower().replace('-', '_').replace(' ', '_')}.jsonl",
    )

    loss_results[name] = result
    loss_rows[name] = rows


print("\n" + "=" * 90)
print("LOSS ABLATION SUMMARY")
print("=" * 90)
print(f"{'Loss':<16} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>10}")
print("-" * 90)

for name, r in sorted(loss_results.items(), key=lambda x: -x[1]["MRR@10"]):
    print(
        f"{name:<16} "
        f"{r['Hit@1'] * 100:>7.2f}% "
        f"{r['Recall@5'] * 100:>9.2f}% "
        f"{r['MRR@10']:>9.4f} "
        f"{r['NDCG@10']:>10.4f}"
    )

best_name, best_result = max(loss_results.items(), key=lambda x: x[1]["MRR@10"])
print("\nBest by MRR@10:", best_name, best_result)


##########################################################################################
ADR-MSE
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_adrmse/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ADR-MSE:   0%|          | 0/390 [00:00<?, ?it/s]


ADR-MSE
n:         390
Hit@1:    0.6667
Recall@5: 0.9154
MRR@10:   0.7734
NDCG@10:  0.8108
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_outputs/rank_adr_mse.jsonl

##########################################################################################
ListwiseKL
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_kl/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

RankNet:   0%|          | 0/390 [00:00<?, ?it/s]


RankNet
n:         390
Hit@1:    0.6718
Recall@5: 0.9179
MRR@10:   0.7747
NDCG@10:  0.8122
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_outputs/rank_ranknet.jsonl

##########################################################################################
Margin-MSE
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_margin/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Margin-MSE:   0%|          | 0/390 [00:00<?, ?it/s]


Margin-MSE
n:         390
Hit@1:    0.6205
Recall@5: 0.8897
MRR@10:   0.7360
NDCG@10:  0.7811
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_outputs/rank_margin_mse.jsonl

LOSS ABLATION SUMMARY
Loss                Hit@1   Recall@5    MRR@10    NDCG@10
------------------------------------------------------------------------------------------
ListwiseKL         69.49%     92.31%    0.7920     0.8272
RankNet            67.18%     91.79%    0.7747     0.8122
ADR-MSE            66.67%     91.54%    0.7734     0.8108
Margin-MSE         62.05%     88.97%    0.7360     0.7811

Best by MRR@10: ListwiseKL {'name': 'ListwiseKL', 'n': 390, 'Hit@1': 0.6948717948717948, 'Recall@5': 0.9230769230769231, 'MRR@10': 0.7920075295075295, 'NDCG@10': 0.8271592466463219}


In [ ]:
# KD thẳng từ MiniLM base, không qua Stage A

BASE_KD_CKPT = MINILM_BASE
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

LOSS_TYPE = "kl"   # "kl" | "ranknet" | "adr_mse"
OUT_KD_DIRECT = f"{ABL}/kd_direct_minilm_base_{LOSS_TYPE}"

ckpt_kd_direct = train_stage_b_ranknet(
    stage_a_checkpoint=BASE_KD_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=OUT_KD_DIRECT,
    loss_type=LOSS_TYPE,
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=1.0,
    patience=2,
    seed=42,
)

print("DONE")
print(f"{LOSS_TYPE}: {ckpt_kd_direct}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: kl | alpha=1.0 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 3.0186 | KD(kl): 3.0186 | CL: 1.5092 | Dev Acc: 0.7541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7541)
Epoch 2/5 | Loss: 2.1074 | KD(kl): 2.1074 | CL: 1.2415 | Dev Acc: 0.8197


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  → Saved best (acc=0.8328)
Epoch 4/5 | Loss: 1.6007 | KD(kl): 1.6007 | CL: 1.2317 | Dev Acc: 0.8426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8426)
Epoch 5/5 | Loss: 1.4734 | KD(kl): 1.4734 | CL: 1.2197 | Dev Acc: 0.8459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8459)
kl done. Best dev acc: 0.8459
DONE
kl: /content/drive/MyDrive/Data/archive/ablation/kd_direct_minilm_base_kl/best


In [ ]:
import gc, torch
from pathlib import Path

KD_DIRECT_MODELS = {
    "KD-direct ADR-MSE": f"{ABL}/kd_direct_minilm_base_adr_mse/best",
    "KD-direct KL":      f"{ABL}/kd_direct_minilm_base_kl/best",
    "KD-direct RankNet": f"{ABL}/kd_direct_minilm_base_ranknet/best",
}

if "dense_pool" not in globals():
    dense_pool, pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )
    print("Dense pre-result:", pre_result)

KD_DIRECT_RANK_OUT = f"{ABL}/kd_direct_minilm_base_rank_outputs"
Path(KD_DIRECT_RANK_OUT).mkdir(parents=True, exist_ok=True)

kd_direct_results = {}

for name, path in KD_DIRECT_MODELS.items():
    if not Path(path).exists():
        print(f"SKIP missing: {name} -> {path}")
        continue

    print(f"\n{'=' * 70}\n{name}\n{path}\n{'=' * 70}")

    result, rows = benchmark_dense_then_rerank(
        reranker_path=path,
        name=name,
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=str(Path(KD_DIRECT_RANK_OUT) / f"{name.replace(' ', '_').replace('-', '_')}.jsonl"),
    )

    kd_direct_results[name] = result

    gc.collect()
    torch.cuda.empty_cache()

print("\nSUMMARY")
print(f"{'Model':<24} {'Hit@1':>8} {'Recall@5':>9} {'MRR@10':>9} {'NDCG@10':>9}")
print("-" * 64)

for name, r in sorted(kd_direct_results.items(), key=lambda x: x[1].get("mrr@10", 0), reverse=True):
    print(
        f"{name:<24} "
        f"{r.get('hit@1', 0):>8.4f} "
        f"{r.get('recall@5', 0):>9.4f} "
        f"{r.get('mrr@10', 0):>9.4f} "
        f"{r.get('ndcg@10', 0):>9.4f}"
    )

SKIP missing: KD-direct ADR-MSE -> /content/drive/MyDrive/Data/archive/ablation/kd_direct_minilm_base_adr_mse/best

KD-direct KL
/content/drive/MyDrive/Data/archive/ablation/kd_direct_minilm_base_kl/best


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

KD-direct KL:   0%|          | 0/390 [00:00<?, ?it/s]


KD-direct KL
n:         390
Hit@1:    0.6821
Recall@5: 0.9179
MRR@10:   0.7855
NDCG@10:  0.8222
Saved: /content/drive/MyDrive/Data/archive/ablation/kd_direct_minilm_base_rank_outputs/KD_direct_KL.jsonl
SKIP missing: KD-direct RankNet -> /content/drive/MyDrive/Data/archive/ablation/kd_direct_minilm_base_ranknet/best

SUMMARY
Model                       Hit@1  Recall@5    MRR@10   NDCG@10
----------------------------------------------------------------
KD-direct KL               0.0000    0.0000    0.0000    0.0000


In [ ]:
# ===== MiniLM base + mMARCO 50k Stage A + Stage B ListwiseKL =====

STAGE_A_MM_OUT = f"{ABL}/stage_a_minilm_mmarco50k_no_synth"
STAGE_B_KL_MM_OUT = f"{ABL}/stage_b_minilm_mmarco50k_kl"

RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

# 1) Stage A: domain no-synth + mMARCO 50k
ckpt_stage_a_mmarco50k = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=MMARCO_DATA,
    dev_path=DOMAIN_DEV,
    output_dir=STAGE_A_MM_OUT,
    base_model=MINILM_BASE,
    domain_upsample=1,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    patience=2,
    seed=42,
)

# 2) Stage B: Listwise KL
ckpt_minilm_mmarco50k_kl = train_stage_b_ranknet(
    stage_a_checkpoint=ckpt_stage_a_mmarco50k,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=STAGE_B_KL_MM_OUT,
    loss_type="kl",
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
)

print("DONE")
print("Stage A:", ckpt_stage_a_mmarco50k)
print("Stage B KL:", ckpt_minilm_mmarco50k_kl)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base: ms-marco-MiniLM-L12-v2 | Seed: 42
Train: 52,328 | Dev: 305
Epoch 1/5 | Loss: 0.4889 | Dev Acc: 0.8262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8262)
Epoch 2/5 | Loss: 0.3699 | Dev Acc: 0.8492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  → Saved best (acc=0.8787)
Epoch 3/5 | Loss: 0.8958 | KD(kl): 1.1874 | CL: 0.2154 | Dev Acc: 0.8820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8820)
Epoch 4/5 | Loss: 0.8319 | KD(kl): 1.0985 | CL: 0.2100 | Dev Acc: 0.8951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8951)
Epoch 5/5 | Loss: 0.8011 | KD(kl): 1.0541 | CL: 0.2109 | Dev Acc: 0.8951
kl done. Best dev acc: 0.8951
DONE
Stage A: /content/drive/MyDrive/Data/archive/ablation/stage_a_minilm_mmarco50k_no_synth/best
Stage B KL: /content/drive/MyDrive/Data/archive/ablation/stage_b_minilm_mmarco50k_kl/best


In [ ]:
CKPT_MINILM_MMARCO50K_KL = "/content/drive/MyDrive/Data/archive/ablation/stage_b_minilm_mmarco50k_kl/best"

result_mmarco50k_kl, rows_mmarco50k_kl = benchmark_dense_then_rerank(
    reranker_path=CKPT_MINILM_MMARCO50K_KL,
    name="MiniLM + mMARCO50k + ListwiseKL",
    dense_pool=dense_pool,
    batch_size=32,
    max_length=512,
    save_rank_path="/content/drive/MyDrive/Data/archive/ablation/rank_minilm_mmarco50k_kl.jsonl",
)

print("\nDONE")
print(result_mmarco50k_kl)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MiniLM + mMARCO50k + ListwiseKL:   0%|          | 0/390 [00:00<?, ?it/s]


MiniLM + mMARCO50k + ListwiseKL
n:         390
Hit@1:    0.6949
Recall@5: 0.9103
MRR@10:   0.7932
NDCG@10:  0.8257
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_minilm_mmarco50k_kl.jsonl

DONE
{'name': 'MiniLM + mMARCO50k + ListwiseKL', 'n': 390, 'Hit@1': 0.6948717948717948, 'Recall@5': 0.9102564102564102, 'MRR@10': 0.7931674806674807, 'NDCG@10': 0.8257461601367493}


In [ ]:
STAGE_A_CKPT = "/content/drive/MyDrive/Data/archive/ablation/stage_a_minilm_colab_no_synth/best"
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

DOMAIN_TRAIN_STAGE_B = DOMAIN_TRAIN
DOMAIN_DEV_STAGE_B = DOMAIN_DEV

LOSS_CFG_KL_ALPHA1 = {
    "epochs": 5,
    "batch_size": 8,
    "lr": 1e-5,
    "max_length": 512,
    "alpha": 1.0,
    "patience": 2,
    "seed": 42,
}

ckpt_kl_alpha1 = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL}/loss_ablation_no_synth_stage_a/stage_b_kl_alpha1",
    loss_type="kl",
    **LOSS_CFG_KL_ALPHA1,
)

print("DONE")
print("KL alpha=1.0:", ckpt_kl_alpha1)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: kl | alpha=1.0 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 2.3329 | KD(kl): 2.3329 | CL: 0.4732 | Dev Acc: 0.8131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8131)
Epoch 2/5 | Loss: 1.7111 | KD(kl): 1.7111 | CL: 0.6582 | Dev Acc: 0.8295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8295)
Epoch 3/5 | Loss: 1.4827 | KD(kl): 1.4827 | CL: 0.6811 | Dev Acc: 0.8262
Epoch 4/5 | Loss: 1.4107 | KD(kl): 1.4107 | CL: 0.7905 | Dev Acc: 0.8295
Early stopping at epoch 4
kl done. Best dev acc: 0.8295
DONE
KL alpha=1.0: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_kl_alpha1/best


In [ ]:
import gc, torch
from pathlib import Path

KL_ALPHA1_MODEL = "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/stage_b_kl_alpha1/best"

# Nếu dense_pool chưa có thì build lại từ embedding model đang dùng
if "dense_pool" not in globals():
    dense_pool, pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )
    print("Dense pre-result:", pre_result)

result_kl_alpha1, rows_kl_alpha1 = benchmark_dense_then_rerank(
    reranker_path=KL_ALPHA1_MODEL,
    name="KL alpha=1.0",
    dense_pool=dense_pool,
    batch_size=32,
    max_length=512,
    save_rank_path="/content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_kl_alpha1.jsonl",
)

print("\nRESULT")
print(f"Hit@1:    {result_kl_alpha1['Hit@1']:.4f}")
print(f"Recall@5: {result_kl_alpha1['Recall@5']:.4f}")
print(f"MRR@10:   {result_kl_alpha1['MRR@10']:.4f}")
print(f"NDCG@10:  {result_kl_alpha1['NDCG@10']:.4f}")

gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

KL alpha=1.0:   0%|          | 0/390 [00:00<?, ?it/s]


KL alpha=1.0
n:         390
Hit@1:    0.6872
Recall@5: 0.9205
MRR@10:   0.7860
NDCG@10:  0.8216
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_no_synth_stage_a/rank_kl_alpha1.jsonl

RESULT


KeyError: 'hit@1'

In [ ]:
H384_STAGE_A_CKPT = MMARCO_BASE

RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

DOMAIN_TRAIN_STAGE_B = DOMAIN_TRAIN
DOMAIN_DEV_STAGE_B = DOMAIN_DEV

ABL_LOSS_H384 = f"{ABL}/loss_ablation_h384_stage_b"

LOSS_CFG_H384 = {
    "epochs": 5,
    "batch_size": 8,
    "lr": 1e-5,
    "max_length": 512,
    "alpha": 0.7,
    "patience": 2,
    "seed": 42,
}

ckpt_h384_adrmse = train_stage_b_ranknet(
    stage_a_checkpoint=H384_STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS_H384}/stage_b_adrmse",
    loss_type="adr_mse",
    **LOSS_CFG_H384,
)

ckpt_h384_kl = train_stage_b_ranknet(
    stage_a_checkpoint=H384_STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS_H384}/stage_b_kl",
    loss_type="kl",
    **LOSS_CFG_H384,
)

ckpt_h384_ranknet = train_stage_b_ranknet(
    stage_a_checkpoint=H384_STAGE_A_CKPT,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS_H384}/stage_b_ranknet",
    loss_type="ranknet",
    **LOSS_CFG_H384,
)

ckpt_h384_margin = train_stage_b_margin_mse(
    base_checkpoint=H384_STAGE_A_CKPT,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV_STAGE_B,
    output_dir=f"{ABL_LOSS_H384}/stage_b_margin",
    domain_train_path=DOMAIN_TRAIN_STAGE_B,
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
    drop_negative_margin=False,
)

print("DONE H384")
print("ADR-MSE :", ckpt_h384_adrmse)
print("KL      :", ckpt_h384_kl)
print("RankNet :", ckpt_h384_ranknet)
print("Margin  :", ckpt_h384_margin)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.4733 | KD(adr_mse): 0.4988 | CL: 0.4136 | Dev Acc: 0.8951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8951)
Epoch 2/5 | Loss: 0.3501 | KD(adr_mse): 0.3986 | CL: 0.2372 | Dev Acc: 0.9016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  -> Saved best (acc=0.8787)
Epoch 2/5 | Loss: 1.1000 | MarginMSE: 1.4308 | CL: 0.3281 | Dev Acc: 0.8951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> Saved best (acc=0.8951)
Epoch 3/5 | Loss: 0.8395 | MarginMSE: 1.0717 | CL: 0.2976 | Dev Acc: 0.8984


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> Saved best (acc=0.8984)
Epoch 4/5 | Loss: 0.6965 | MarginMSE: 0.8754 | CL: 0.2790 | Dev Acc: 0.8885
Epoch 5/5 | Loss: 0.6230 | MarginMSE: 0.7712 | CL: 0.2770 | Dev Acc: 0.8885
Early stopping at epoch 5
margin_mse done. Best dev acc: 0.8984
DONE H384
ADR-MSE : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_adrmse/best
KL      : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_kl/best
RankNet : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_ranknet/best
Margin  : /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_margin/best


In [ ]:
# ===== Benchmark H384 Stage-B 4 loss models =====

H384_LOSS_MODELS = {
    "H384 ADR-MSE": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_adrmse/best",
    "H384 ListwiseKL": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_kl/best",
    "H384 RankNet": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_ranknet/best",
    "H384 Margin-MSE": "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_margin/best",
}

H384_RANK_OUT_DIR = "/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/rank_outputs"
Path(H384_RANK_OUT_DIR).mkdir(parents=True, exist_ok=True)

h384_results = {}
h384_rows = {}

for name, ckpt in H384_LOSS_MODELS.items():
    print("\n" + "#" * 90)
    print(name)
    print(ckpt)
    print("#" * 90)

    result, rows = benchmark_dense_then_rerank(
        reranker_path=ckpt,
        name=name,
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=f"{H384_RANK_OUT_DIR}/rank_{name.lower().replace('-', '_').replace(' ', '_')}.jsonl",
    )

    h384_results[name] = result
    h384_rows[name] = rows


print("\n" + "=" * 90)
print("H384 LOSS ABLATION SUMMARY")
print("=" * 90)
print(f"{'Loss':<20} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>10}")
print("-" * 90)

for name, r in sorted(h384_results.items(), key=lambda x: -x[1]["MRR@10"]):
    print(
        f"{name:<20} "
        f"{r['Hit@1'] * 100:>7.2f}% "
        f"{r['Recall@5'] * 100:>9.2f}% "
        f"{r['MRR@10']:>9.4f} "
        f"{r['NDCG@10']:>10.4f}"
    )

best_h384_name, best_h384_result = max(h384_results.items(), key=lambda x: x[1]["MRR@10"])
print("\nBest H384 by MRR@10:", best_h384_name, best_h384_result)


##########################################################################################
H384 ADR-MSE
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_adrmse/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

H384 ADR-MSE:   0%|          | 0/390 [00:00<?, ?it/s]


H384 ADR-MSE
n:         390
Hit@1:    0.8308
Recall@5: 0.9615
MRR@10:   0.8862
NDCG@10:  0.9069
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/rank_outputs/rank_h384_adr_mse.jsonl

##########################################################################################
H384 ListwiseKL
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_kl/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

H384 RankNet:   0%|          | 0/390 [00:00<?, ?it/s]


H384 RankNet
n:         390
Hit@1:    0.8231
Recall@5: 0.9615
MRR@10:   0.8842
NDCG@10:  0.9050
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/rank_outputs/rank_h384_ranknet.jsonl

##########################################################################################
H384 Margin-MSE
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_margin/best
##########################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

H384 Margin-MSE:   0%|          | 0/390 [00:00<?, ?it/s]


H384 Margin-MSE
n:         390
Hit@1:    0.8256
Recall@5: 0.9641
MRR@10:   0.8832
NDCG@10:  0.9039
Saved: /content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/rank_outputs/rank_h384_margin_mse.jsonl

H384 LOSS ABLATION SUMMARY
Loss                    Hit@1   Recall@5    MRR@10    NDCG@10
------------------------------------------------------------------------------------------
H384 ADR-MSE           83.08%     96.15%    0.8862     0.9069
H384 RankNet           82.31%     96.15%    0.8842     0.9050
H384 Margin-MSE        82.56%     96.41%    0.8832     0.9039
H384 ListwiseKL        82.05%     96.67%    0.8815     0.9024

Best H384 by MRR@10: H384 ADR-MSE {'name': 'H384 ADR-MSE', 'n': 390, 'Hit@1': 0.8307692307692308, 'Recall@5': 0.9615384615384616, 'MRR@10': 0.8862494912494913, 'NDCG@10': 0.9068588053654117}


In [ ]:
import os, json, gc
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# Config
# =========================
H384_BASE = MMARCO_BASE
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

ALL_CHUNK_DIR = Path("/content/drive/MyDrive/Data/chunk_outputs_finals")

LOSS_TYPE = "adr_mse"
PRUNED_H384_BASE = "/content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old"
OUT_STAGE_B = f"{ABL}/stage_b_pruned_h384_{LOSS_TYPE}_alpha07"
EXTRA_COMMON = 30000
MAX_LEN_PRUNE = 512

# =========================
# 1. Build kept_ids giống code cũ
# =========================
tok_prune = AutoTokenizer.from_pretrained(H384_BASE)
print(f"Original vocab: {tok_prune.vocab_size}")

token_counter = Counter()

def add_text(t):
    if not t or not str(t).strip():
        return
    ids = tok_prune(
        str(t),
        add_special_tokens=False,
        max_length=MAX_LEN_PRUNE,
        truncation=True,
    )["input_ids"]
    token_counter.update(ids)

# Tất cả chunks: ưu tiên biến texts nếu notebook benchmark đã load sẵn
n_chunks = 0
for doc_dir in ALL_CHUNK_DIR.iterdir():
    if not doc_dir.is_dir():
        continue
    for cf in doc_dir.glob("*.json"):
        recs = json.load(open(cf, encoding="utf-8"))
        if not isinstance(recs, list):
            continue
        for rec in recs:
            add_text(str(rec.get("page_content", "")))
            n_chunks += 1

# Tất cả queries từ retrieve_rerank_991_logit
with open(RERANK_LOGIT, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            add_text(rec.get("question") or rec.get("query") or "")

print(f"Chunks: {n_chunks} | Unique tokens: {len(token_counter)}")

special_ids = set(tok_prune.all_special_ids)
kept_ids = set(special_ids)

for tid in token_counter:
    kept_ids.add(tid)

for tid in range(min(EXTRA_COMMON, tok_prune.vocab_size)):
    kept_ids.add(tid)

kept_ids = sorted(kept_ids)
old_to_new = {old: new for new, old in enumerate(kept_ids)}

print(
    f"Tokens giữ: {len(kept_ids)}/{tok_prune.vocab_size} "
    f"(giảm {(1 - len(kept_ids) / tok_prune.vocab_size) * 100:.1f}%)"
)

# =========================
# 2. Save pruned model/tokenizer giống code cũ
# =========================
model_p = AutoModelForSequenceClassification.from_pretrained(H384_BASE)
orig_params = sum(p.numel() for p in model_p.parameters())

embed = model_p.roberta.embeddings.word_embeddings
old_w = embed.weight.data
hidden = old_w.shape[1]

new_w = old_w[kept_ids].clone()
new_embed = torch.nn.Embedding(
    len(kept_ids),
    hidden,
    padding_idx=old_to_new.get(tok_prune.pad_token_id, 0),
)
new_embed.weight.data = new_w

model_p.roberta.embeddings.word_embeddings = new_embed
model_p.config.vocab_size = len(kept_ids)

new_params = sum(p.numel() for p in model_p.parameters())
print(
    f"Params: {orig_params/1e6:.1f}M -> {new_params/1e6:.1f}M "
    f"(giảm {(1 - new_params / orig_params) * 100:.1f}%)"
)

os.makedirs(PRUNED_H384_BASE, exist_ok=True)

tok_prune.save_pretrained(PRUNED_H384_BASE)

tj = os.path.join(PRUNED_H384_BASE, "tokenizer.json")
with open(tj, encoding="utf-8") as f:
    tdata = json.load(f)

if tdata["model"]["type"] == "Unigram":
    ov = tdata["model"]["vocab"]
    tdata["model"]["vocab"] = [ov[o] for o in kept_ids]
    if "unk_id" in tdata["model"]:
        tdata["model"]["unk_id"] = old_to_new.get(tdata["model"]["unk_id"], 0)

if "added_tokens" in tdata:
    for at in tdata["added_tokens"]:
        if at["id"] in old_to_new:
            at["id"] = old_to_new[at["id"]]

with open(tj, "w", encoding="utf-8") as f:
    json.dump(tdata, f, ensure_ascii=False)

model_p.save_pretrained(PRUNED_H384_BASE)

del model_p
gc.collect()
torch.cuda.empty_cache()

print(f"Saved pruned base -> {PRUNED_H384_BASE}")

# =========================
# 3. Train thẳng Stage B alpha=0.7
# =========================
ckpt_pruned_h384_adrmse_alpha07 = train_stage_b_ranknet(
    stage_a_checkpoint=PRUNED_H384_BASE,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=OUT_STAGE_B,
    loss_type=LOSS_TYPE,
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
)

print("DONE")
print(f"Pruned H384 Stage B {LOSS_TYPE} alpha=0.7:", ckpt_pruned_h384_adrmse_alpha07)

Original vocab: 250002
Chunks: 2317 | Unique tokens: 10972
Tokens giữ: 36329/250002 (giảm 85.5%)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Params: 117.6M -> 35.6M (giảm 69.7%)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved pruned base -> /content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old


...

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.4731 | KD(adr_mse): 0.4985 | CL: 0.4137 | Dev Acc: 0.8951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8951)
Epoch 2/5 | Loss: 0.3503 | KD(adr_mse): 0.3986 | CL: 0.2376 | Dev Acc: 0.8918
Epoch 3/5 | Loss: 0.3173 | KD(adr_mse): 0.3788 | CL: 0.1739 | Dev Acc: 0.8984


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8984)
Epoch 4/5 | Loss: 0.3022 | KD(adr_mse): 0.3699 | CL: 0.1442 | Dev Acc: 0.8984
Epoch 5/5 | Loss: 0.2905 | KD(adr_mse): 0.3610 | CL: 0.1261 | Dev Acc: 0.8984
Early stopping at epoch 5
adr_mse done. Best dev acc: 0.8984
DONE
Pruned H384 Stage B adr_mse alpha=0.7: /content/drive/MyDrive/Data/archive/ablation/stage_b_pruned_h384_adr_mse_alpha07/best


In [ ]:
import gc, torch
from pathlib import Path

PRUNED_H384_ADRMSE = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"

if "ckpt_pruned_h384_adrmse_alpha07" in globals():
    PRUNED_H384_ADRMSE = ckpt_pruned_h384_adrmse_alpha07

print("Benchmark model:", PRUNED_H384_ADRMSE)

if "dense_pool" not in globals():
    dense_pool, pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )
    print("Dense pre-result:", pre_result)

result_pruned_h384_adrmse, rows_pruned_h384_adrmse = benchmark_dense_then_rerank(
    reranker_path=PRUNED_H384_ADRMSE,
    name="Pruned H384 ADR-MSE alpha=0.7",
    dense_pool=dense_pool,
    batch_size=32,
    max_length=512,
    save_rank_path=f"{ABL}/rank_pruned_h384_adrmse_alpha07.jsonl",
)

print("\nRESULT")
print(f"Hit@1:    {result_pruned_h384_adrmse.get('Hit@1', result_pruned_h384_adrmse.get('hit@1')):.4f}")
print(f"Recall@5: {result_pruned_h384_adrmse.get('Recall@5', result_pruned_h384_adrmse.get('recall@5')):.4f}")
print(f"MRR@10:   {result_pruned_h384_adrmse.get('MRR@10', result_pruned_h384_adrmse.get('mrr@10')):.4f}")
print(f"NDCG@10:  {result_pruned_h384_adrmse.get('NDCG@10', result_pruned_h384_adrmse.get('ndcg@10')):.4f}")

gc.collect()
torch.cuda.empty_cache()

Benchmark model: /content/drive/MyDrive/Data/archive/ablation/stage_b_pruned_h384_adr_mse_alpha07/best


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Pruned H384 ADR-MSE alpha=0.7:   0%|          | 0/390 [00:00<?, ?it/s]


Pruned H384 ADR-MSE alpha=0.7
n:         390
Hit@1:    0.8282
Recall@5: 0.9590
MRR@10:   0.8844
NDCG@10:  0.9051
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_pruned_h384_adrmse_alpha07.jsonl

RESULT
Hit@1:    0.8282
Recall@5: 0.9590
MRR@10:   0.8844
NDCG@10:  0.9051


In [ ]:
import gc, torch
from pathlib import Path

VIRANKER = "namdp-ptit/ViRanker"
BGE_RERANKER = "/content/drive/MyDrive/Data/archive/bge-reranker-v2-m3"

MMARCO_117M_BASE = MMARCO_BASE
PRUNED_BASE = "/content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old"

BASE_TEACHER_MODELS = {
    "ViRanker": VIRANKER,
    "BGE-reranker-v2-m3 teacher": BGE_RERANKER,
    "mMARCO H384 117M base": MMARCO_117M_BASE,
    "Pruned H384 base": PRUNED_BASE,
}

if "dense_pool" not in globals():
    dense_pool, pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )
    print("Dense pre-result:", pre_result)

RANK_OUT_DIR = f"{ABL}/rank_teacher_base_models"
Path(RANK_OUT_DIR).mkdir(parents=True, exist_ok=True)

teacher_base_results = {}

for name, path in BASE_TEACHER_MODELS.items():
    if not Path(path).exists():
        print(f"SKIP missing: {name} -> {path}")
        continue

    print(f"\n{'=' * 80}")
    print(name)
    print(path)
    print(f"{'=' * 80}")

    result, rows = benchmark_dense_then_rerank(
        reranker_path=path,
        name=name,
        dense_pool=dense_pool,
        batch_size=16 if "BGE" in name or "ViRanker" in name else 32,
        max_length=512,
        save_rank_path=str(Path(RANK_OUT_DIR) / f"{name.replace(' ', '_').replace('/', '_')}.jsonl"),
    )

    teacher_base_results[name] = result

    gc.collect()
    torch.cuda.empty_cache()

def metric(r, a, b):
    return r[a] if a in r else r[b]

print("\nSUMMARY")
print(f"{'Model':<30} {'Hit@1':>8} {'Recall@5':>9} {'MRR@10':>9} {'NDCG@10':>9}")
print("-" * 76)

for name, r in sorted(
    teacher_base_results.items(),
    key=lambda x: metric(x[1], "MRR@10", "mrr@10"),
    reverse=True,
):
    print(
        f"{name:<30} "
        f"{metric(r, 'Hit@1', 'hit@1'):>8.4f} "
        f"{metric(r, 'Recall@5', 'recall@5'):>9.4f} "
        f"{metric(r, 'MRR@10', 'mrr@10'):>9.4f} "
        f"{metric(r, 'NDCG@10', 'ndcg@10'):>9.4f}"
    )

SKIP missing: ViRanker -> namdp-ptit/ViRanker

BGE-reranker-v2-m3 teacher
/content/drive/MyDrive/Data/archive/bge-reranker-v2-m3


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE-reranker-v2-m3 teacher:   0%|          | 0/390 [00:00<?, ?it/s]


BGE-reranker-v2-m3 teacher
n:         390
Hit@1:    0.8256
Recall@5: 0.9615
MRR@10:   0.8862
NDCG@10:  0.9068
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_teacher_base_models/BGE-reranker-v2-m3_teacher.jsonl

mMARCO H384 117M base
/content/drive/MyDrive/Data/archive/mmarco-mMiniLMv2-L12-H384-v1


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

mMARCO H384 117M base:   0%|          | 0/390 [00:00<?, ?it/s]


mMARCO H384 117M base
n:         390
Hit@1:    0.7615
Recall@5: 0.9410
MRR@10:   0.8442
NDCG@10:  0.8732
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_teacher_base_models/mMARCO_H384_117M_base.jsonl

Pruned H384 base
/content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Pruned H384 base:   0%|          | 0/390 [00:00<?, ?it/s]


Pruned H384 base
n:         390
Hit@1:    0.7564
Recall@5: 0.9385
MRR@10:   0.8423
NDCG@10:  0.8710
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_teacher_base_models/Pruned_H384_base.jsonl

SUMMARY
Model                             Hit@1  Recall@5    MRR@10   NDCG@10
----------------------------------------------------------------------------
BGE-reranker-v2-m3 teacher       0.8256    0.9615    0.8862    0.9068
mMARCO H384 117M base            0.7615    0.9410    0.8442    0.8732
Pruned H384 base                 0.7564    0.9385    0.8423    0.8710


In [ ]:
import gc, torch

VIRANKER_HF = "namdp-ptit/ViRanker"

# Build dense pool nếu chưa có
if "dense_pool" not in globals():
    dense_pool, pre_result = build_dense_topk_pool_512(
        embed_model_path=EMB_FT,
        top_k=TOP_K_DENSE,
        dim=EMB_DIM,
        batch_size=64,
    )
    print("Dense pre-result:", pre_result)

result_viranker_hf, rows_viranker_hf = benchmark_dense_then_rerank(
    reranker_path=VIRANKER_HF,
    name="ViRanker HF",
    dense_pool=dense_pool,
    batch_size=16,
    max_length=512,
    save_rank_path=f"{ABL}/rank_viranker_hf.jsonl",
)

print("\nRESULT")
print(f"Hit@1:    {result_viranker_hf.get('Hit@1', result_viranker_hf.get('hit@1')):.4f}")
print(f"Recall@5: {result_viranker_hf.get('Recall@5', result_viranker_hf.get('recall@5')):.4f}")
print(f"MRR@10:   {result_viranker_hf.get('MRR@10', result_viranker_hf.get('mrr@10')):.4f}")
print(f"NDCG@10:  {result_viranker_hf.get('NDCG@10', result_viranker_hf.get('ndcg@10')):.4f}")

gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ViRanker HF:   0%|          | 0/390 [00:00<?, ?it/s]


ViRanker HF
n:         390
Hit@1:    0.7872
Recall@5: 0.9564
MRR@10:   0.8593
NDCG@10:  0.8844
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_viranker_hf.jsonl

RESULT
Hit@1:    0.7872
Recall@5: 0.9564
MRR@10:   0.8593
NDCG@10:  0.8844


In [ ]:
# ===== ƯU TIÊN 1: Bảng Chất lượng vs Tốc độ =====
import os, gc, time, json
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RERANK_MODELS = {
    # --- Teacher / baseline lớn ---
    "BGE-reranker-v2-m3 (teacher)": "/content/drive/MyDrive/Data/archive/bge-reranker-v2-m3",
    "ViRanker":                     "namdp-ptit/ViRanker",

    # --- Student của em ---
    "MiniLM-L12 + StageA+B":  f"{ABL}/loss_ablation_no_synth_stage_a/stage_b_adrmse/best",
    "H384 + StageB":          f"{ABL}/loss_ablation_h384_stage_b/stage_b_adrmse/best",
    "Pruned H384 + StageB":   f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best",
}

N_WARMUP = 3
N_REPEAT = 30
TOP_K    = 20          # rerank top-20 như pipeline thật
MAX_LEN  = 512


def dir_size_mb(path):
    """Dung lượng model trên đĩa (MB). HF hub id -> trả None."""
    p = Path(path)
    if not p.exists():
        return None
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1024 / 1024


@torch.no_grad()
def measure_latency(model, tokenizer, query, docs, max_length=MAX_LEN):
    """ms để rerank 1 query với len(docs) candidates (1 batch)."""
    model.eval()

    def _one_pass():
        enc = tokenizer(
            [query] * len(docs), docs,
            padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(model.device)
        _ = model(**enc).logits

    for _ in range(N_WARMUP):
        _one_pass()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        _one_pass()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    return float(np.median(times)), float(np.percentile(times, 95))


# Lấy 1 query thật + 20 candidate thật từ dense_pool
if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64,
    )

sample   = dense_pool[0]
q_sample = sample["question"]
d_sample = sample["candidate_texts"][:TOP_K]
print(f"Query mẫu: {q_sample[:60]}... | {len(d_sample)} candidates\n")

speed_rows = []

for name, path in RERANK_MODELS.items():
    if Path(path).exists() is False and "/" not in path.replace("namdp-ptit/", ""):
        print(f"SKIP missing: {name} -> {path}")
        continue

    print(f"{'='*70}\n{name}")
    try:
        tok = AutoTokenizer.from_pretrained(path, use_fast=False)
        model = AutoModelForSequenceClassification.from_pretrained(path).to(device)
    except Exception as e:
        print(f"  LỖI load ({type(e).__name__}) -> bỏ qua")
        continue

    n_params = sum(p.numel() for p in model.parameters())
    size_mb  = dir_size_mb(path)
    med_ms, p95_ms = measure_latency(model, tok, q_sample, d_sample)

    speed_rows.append({
        "name":       name,
        "params_M":   n_params / 1e6,
        "size_MB":    size_mb,
        "ms_per_q":   med_ms,
        "p95_ms":     p95_ms,
        "ms_per_doc": med_ms / len(d_sample),
    })

    print(f"  Params      : {n_params/1e6:7.1f} M")
    print(f"  Size        : {size_mb:7.1f} MB" if size_mb else "  Size        :     n/a (HF hub)")
    print(f"  Rerank@{TOP_K}   : {med_ms:7.1f} ms/query (p95 {p95_ms:.1f})")
    print(f"  Per doc     : {med_ms/len(d_sample):7.2f} ms")

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()


QUALITY = {}
for src in ["loss_results", "h384_results", "teacher_base_results"]:
    if src in globals():
        for k, v in globals()[src].items():
            QUALITY[k] = v.get("MRR@10", v.get("mrr@10"))
if "result_pruned_h384_adrmse" in globals():
    r = result_pruned_h384_adrmse
    QUALITY["Pruned H384 + StageB"] = r.get("MRR@10", r.get("mrr@10"))

def lookup_mrr(name):
    if name in QUALITY:
        return QUALITY[name]
    for k, v in QUALITY.items():          # khớp lỏng theo từ khoá
        key = name.split("(")[0].strip().lower()
        if key[:6] in k.lower():
            return v
    return None

print("\n" + "=" * 96)
print("BẢNG CHẤT LƯỢNG vs TỐC ĐỘ")
print("=" * 96)
print(f"{'Model':<32} {'Params(M)':>10} {'Size(MB)':>10} {'ms/query':>10} {'ms/doc':>9} {'MRR@10':>9} {'Speedup':>9}")
print("-" * 96)

teacher_ms = next((r["ms_per_q"] for r in speed_rows if "BGE" in r["name"]), None)

for r in sorted(speed_rows, key=lambda x: x["ms_per_q"]):
    mrr     = lookup_mrr(r["name"])
    speedup = teacher_ms / r["ms_per_q"] if teacher_ms else None
    print(
        f"{r['name']:<32} "
        f"{r['params_M']:>10.1f} "
        f"{(f'{r[chr(34)+chr(34)]}' if False else (f'{r[  'size_MB' ]:.1f}' if r['size_MB'] else 'n/a')):>10} "
        f"{r['ms_per_q']:>10.1f} "
        f"{r['ms_per_doc']:>9.2f} "
        f"{(f'{mrr:.4f}' if mrr is not None else '-'):>9} "
        f"{(f'{speedup:.1f}x' if speedup else '-'):>9}"
    )

with open(f"{ABL}/speed_benchmark.json", "w") as f:
    json.dump(speed_rows, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: {ABL}/speed_benchmark.json")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Raw embedding shape: (1, 1024)
Encoding corpus...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

  Params      :    33.4 M
  Size        :   128.0 MB
  Rerank@20   :    53.9 ms/query (p95 54.2)
  Per doc     :    2.70 ms
H384 + StageB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Params      :   117.6 M
  Size        :   465.1 MB
  Rerank@20   :    50.8 ms/query (p95 53.8)
  Per doc     :    2.54 ms
Pruned H384 + StageB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Params      :    35.6 M
  Size        :   138.3 MB
  Rerank@20   :    51.0 ms/query (p95 51.4)
  Per doc     :    2.55 ms

BẢNG CHẤT LƯỢNG vs TỐC ĐỘ
Model                             Params(M)   Size(MB)   ms/query    ms/doc    MRR@10   Speedup
------------------------------------------------------------------------------------------------
H384 + StageB                         117.6      465.1       50.8      2.54         -      7.6x
Pruned H384 + StageB                   35.6      138.3       51.0      2.55         -      7.6x
MiniLM-L12 + StageA+B                  33.4      128.0       53.9      2.70         -      7.1x
BGE-reranker-v2-m3 (teacher)          567.8     2187.0      385.5     19.28         -      1.0x

Đã lưu: /content/drive/MyDrive/Data/archive/ablation/speed_benchmark.json


In [ ]:
# ===== TEST RIÊNG ViRanker: chẩn đoán + benchmark =====
import gc, time, traceback
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

VIRANKER_ID = "namdp-ptit/ViRanker"

# ---------- BƯỚC 1: Xem config trước khi load model ----------
print("=" * 70)
print("BƯỚC 1: Đọc config")
print("=" * 70)
try:
    cfg = AutoConfig.from_pretrained(VIRANKER_ID)
    print("model_type              :", cfg.model_type)
    print("num_labels              :", cfg.num_labels)
    print("vocab_size              :", cfg.vocab_size)
    print("max_position_embeddings :", cfg.max_position_embeddings)   # ⚠️ QUAN TRỌNG
    print("hidden_size             :", cfg.hidden_size)
    print("num_hidden_layers       :", cfg.num_hidden_layers)
except Exception:
    traceback.print_exc()

# PhoBERT: max_position_embeddings = 258 -> max_length THỰC TẾ chỉ 256
SAFE_MAX_LEN = min(512, getattr(cfg, "max_position_embeddings", 512) - 2)
print(f"\n=> Dùng max_length = {SAFE_MAX_LEN}")


# ---------- BƯỚC 2: Thử load tokenizer theo nhiều cách ----------
print("\n" + "=" * 70)
print("BƯỚC 2: Load tokenizer")
print("=" * 70)

tok = None
for desc, kwargs in [
    ("use_fast=False",                    dict(use_fast=False)),
    ("use_fast=True",                     dict(use_fast=True)),
    ("use_fast=False + trust_remote_code", dict(use_fast=False, trust_remote_code=True)),
]:
    try:
        tok = AutoTokenizer.from_pretrained(VIRANKER_ID, **kwargs)
        print(f"✓ OK với {desc}  -> {type(tok).__name__}")
        break
    except Exception as e:
        print(f"✗ Lỗi với {desc}: {type(e).__name__}: {str(e)[:160]}")

if tok is None:
    print("\n⚠️ Tokenizer load thất bại hết. Thử cài thêm rồi RESTART runtime:")
    print("   !pip install -q sentencepiece protobuf")
    raise SystemExit


# ---------- BƯỚC 3: Load model ----------
print("\n" + "=" * 70)
print("BƯỚC 3: Load model")
print("=" * 70)
model = AutoModelForSequenceClassification.from_pretrained(VIRANKER_ID).to(device)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded | params = {n_params/1e6:.1f}M | num_labels = {model.config.num_labels}")


# ---------- BƯỚC 4: Test forward 1 cặp ----------
print("\n" + "=" * 70)
print("BƯỚC 4: Forward thử")
print("=" * 70)
with torch.no_grad():
    enc = tok(["Câu hỏi thử nghiệm là gì?"], ["Đây là một đoạn văn bản mẫu."],
              padding=True, truncation=True, max_length=SAFE_MAX_LEN,
              return_tensors="pt").to(device)
    out = model(**enc).logits
print("✓ logits shape:", tuple(out.shape), "| value:", out.flatten().tolist())


# ---------- BƯỚC 5: Benchmark cùng điều kiện với các model khác ----------
print("\n" + "=" * 70)
print("BƯỚC 5: Benchmark tốc độ")
print("=" * 70)

if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64)

sample = dense_pool[0]
q_sample = sample["question"]
d_sample = sample["candidate_texts"][:20]

@torch.no_grad()
def _pass(max_len):
    enc = tok([q_sample] * len(d_sample), d_sample,
              padding=True, truncation=True, max_length=max_len,
              return_tensors="pt").to(device)
    return model(**enc).logits

for _ in range(3):
    _pass(SAFE_MAX_LEN)
if torch.cuda.is_available():
    torch.cuda.synchronize()

times = []
for _ in range(30):
    t0 = time.perf_counter()
    _pass(SAFE_MAX_LEN)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

med, p95 = float(np.median(times)), float(np.percentile(times, 95))
print(f"Params    : {n_params/1e6:.1f} M")
print(f"max_length: {SAFE_MAX_LEN}  (⚠️ khác 512 nếu PhoBERT)")
print(f"Rerank@20 : {med:.1f} ms/query (p95 {p95:.1f})")
print(f"Per doc   : {med/len(d_sample):.2f} ms")


# ---------- BƯỚC 6: Đo chất lượng ----------
print("\n" + "=" * 70)
print("BƯỚC 6: MRR@10 / NDCG@10")
print("=" * 70)
result_viranker, rows_viranker = benchmark_dense_then_rerank(
    reranker_path=VIRANKER_ID,
    name="ViRanker",
    dense_pool=dense_pool,
    batch_size=16,
    max_length=SAFE_MAX_LEN,     # dùng đúng độ dài hợp lệ
)

del model, tok
gc.collect()
torch.cuda.empty_cache()

BƯỚC 1: Đọc config


config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

model_type              : xlm-roberta
num_labels              : 1
vocab_size              : 250002
max_position_embeddings : 8194
hidden_size             : 1024
num_hidden_layers       : 24

=> Dùng max_length = 512

BƯỚC 2: Load tokenizer


tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✓ Loaded | params = 567.8M | num_labels = 1

BƯỚC 4: Forward thử
✓ logits shape: (1, 1) | value: [-13.188087463378906]

BƯỚC 5: Benchmark tốc độ
Params    : 567.8 M
max_length: 512  (⚠️ khác 512 nếu PhoBERT)
Rerank@20 : 385.7 ms/query (p95 386.6)
Per doc   : 19.28 ms

BƯỚC 6: MRR@10 / NDCG@10


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ViRanker:   0%|          | 0/390 [00:00<?, ?it/s]


ViRanker
n:         390
Hit@1:    0.7872
Recall@5: 0.9564
MRR@10:   0.8593
NDCG@10:  0.8844


In [ ]:
# ===== ĐO ViRanker: params + size + tốc độ + chất lượng =====
import gc, time, numpy as np, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

VIRANKER_ID = "namdp-ptit/ViRanker"
N_WARMUP, N_REPEAT, TOP_K, MAX_LEN = 3, 30, 20, 512

# --- dense_pool ---
if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64)

sample   = dense_pool[0]
q_sample = sample["question"]
d_sample = sample["candidate_texts"][:TOP_K]

# --- load ---
tok   = AutoTokenizer.from_pretrained(VIRANKER_ID, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(VIRANKER_ID).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
size_mb  = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2  # size thực từ weights

print(f"Params      : {n_params/1e6:.1f} M")
print(f"Size (fp32) : {size_mb:.1f} MB")
print(f"num_labels  : {model.config.num_labels} | max_pos: {model.config.max_position_embeddings}")

# --- latency ---
@torch.no_grad()
def _pass():
    enc = tok([q_sample] * len(d_sample), d_sample,
              padding=True, truncation=True, max_length=MAX_LEN,
              return_tensors="pt").to(device)
    return model(**enc).logits

for _ in range(N_WARMUP):
    _pass()
if torch.cuda.is_available():
    torch.cuda.synchronize()

times = []
for _ in range(N_REPEAT):
    t0 = time.perf_counter()
    _pass()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

med, p95 = float(np.median(times)), float(np.percentile(times, 95))
print(f"Rerank@{TOP_K}   : {med:.1f} ms/query (p95 {p95:.1f})")
print(f"Per doc     : {med/len(d_sample):.2f} ms")

del model, tok
gc.collect(); torch.cuda.empty_cache()

# --- chất lượng ---
result_viranker, rows_viranker = benchmark_dense_then_rerank(
    reranker_path=VIRANKER_ID,
    name="ViRanker",
    dense_pool=dense_pool,
    batch_size=16,
    max_length=MAX_LEN,
    save_rank_path=f"{ABL}/rank_viranker.jsonl",
)

# --- ghi vào speed_rows để bảng tổng hợp có ViRanker ---
if "speed_rows" not in globals():
    speed_rows = []
speed_rows = [r for r in speed_rows if r["name"] != "ViRanker"]   # tránh trùng
speed_rows.append({
    "name":       "ViRanker",
    "params_M":   n_params / 1e6,
    "size_MB":    size_mb,
    "ms_per_q":   med,
    "p95_ms":     p95,
    "ms_per_doc": med / len(d_sample),
    "MRR@10":     result_viranker["MRR@10"],
    "NDCG@10":    result_viranker["NDCG@10"],
    "Hit@1":      result_viranker["Hit@1"],
    "Recall@5":   result_viranker["Recall@5"],
})

print("\n" + "=" * 60)
print(f"ViRanker | {n_params/1e6:.1f}M | {med:.1f} ms | MRR@10 = {result_viranker['MRR@10']:.4f}")
print("=" * 60)

gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Params      : 567.8 M
Size (fp32) : 2165.8 MB
num_labels  : 1 | max_pos: 8194
Rerank@20   : 385.7 ms/query (p95 388.6)
Per doc     : 19.28 ms


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ViRanker:   0%|          | 0/390 [00:00<?, ?it/s]


ViRanker
n:         390
Hit@1:    0.7872
Recall@5: 0.9564
MRR@10:   0.8593
NDCG@10:  0.8844
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_viranker.jsonl

ViRanker | 567.8M | 385.7 ms | MRR@10 = 0.8593


In [ ]:
# ===== ALPHA SWEEP: Pruned H384 + ADR-MSE =====
import gc, torch, json
from pathlib import Path

PRUNED_BASE  = "/content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old"
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"
SWEEP_DIR    = f"{ABL}/alpha_sweep_pruned_adrmse"
Path(SWEEP_DIR).mkdir(parents=True, exist_ok=True)

ALPHAS = [0.5, 1.0]        # 0.7 đã có rồi

sweep_ckpts = {}

for a in ALPHAS:
    tag = f"alpha{str(a).replace('.', '')}"
    out = f"{SWEEP_DIR}/stage_b_adrmse_{tag}"
    print("\n" + "#" * 80)
    print(f"ALPHA = {a}")
    print("#" * 80)

    ckpt = train_stage_b_ranknet(
        stage_a_checkpoint=PRUNED_BASE,   # không qua Stage A — giống hệt cell 18
        rerank_path=RERANK_LOGIT,
        domain_train_path=DOMAIN_TRAIN,
        dev_path=DOMAIN_DEV,
        output_dir=out,
        loss_type="adr_mse",
        epochs=5,
        batch_size=8,
        lr=1e-5,
        max_length=512,
        alpha=a,
        patience=2,
        seed=42,
    )
    sweep_ckpts[a] = ckpt
    gc.collect(); torch.cuda.empty_cache()

print("\nDONE")
for a, c in sweep_ckpts.items():
    print(f"  alpha={a}: {c}")


################################################################################
ALPHA = 0.5
################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.5 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.4572 | KD(adr_mse): 0.5162 | CL: 0.3981 | Dev Acc: 0.8951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8951)
Epoch 2/5 | Loss: 0.3065 | KD(adr_mse): 0.4256 | CL: 0.1874 | Dev Acc: 0.8787
Epoch 3/5 | Loss: 0.2653 | KD(adr_mse): 0.4143 | CL: 0.1163 | Dev Acc: 0.8885
Early stopping at epoch 3
adr_mse done. Best dev acc: 0.8951

################################################################################
ALPHA = 1.0
################################################################################


...

  → Saved best (acc=0.8951)
Epoch 3/5 | Loss: 0.3598 | KD(adr_mse): 0.3598 | CL: 0.5822 | Dev Acc: 0.9016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.9016)
Epoch 4/5 | Loss: 0.3473 | KD(adr_mse): 0.3473 | CL: 0.6266 | Dev Acc: 0.9049


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.9049)


In [ ]:
# ===== BENCHMARK ALPHA SWEEP: Pruned H384 + ADR-MSE =====
import gc, json, torch
from pathlib import Path

SWEEP_DIR = f"{ABL}/alpha_sweep_pruned_adrmse"

ALPHA_MODELS = {
    0.3: f"{SWEEP_DIR}/stage_b_adrmse_alpha03/best",
    0.5: f"{SWEEP_DIR}/stage_b_adrmse_alpha05/best",
    0.7: f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best",   # đã train từ trước
    1.0: f"{SWEEP_DIR}/stage_b_adrmse_alpha10/best",
}

if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64)

alpha_results = {}

for a, path in sorted(ALPHA_MODELS.items()):
    if not Path(path).exists():
        print(f"⚠️  SKIP missing: alpha={a} -> {path}")
        continue

    print("\n" + "=" * 74)
    print(f"ALPHA = {a}   (w_KD={a:.1f}, w_CL={1-a:.1f})")
    print(path)
    print("=" * 74)

    result, _ = benchmark_dense_then_rerank(
        reranker_path=path,
        name=f"Pruned ADR-MSE alpha={a}",
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=f"{SWEEP_DIR}/rank_alpha{str(a).replace('.','')}.jsonl",
    )

    alpha_results[a] = result
    gc.collect()
    torch.cuda.empty_cache()


# ===== BẢNG TỔNG HỢP =====
print("\n" + "=" * 84)
print("ALPHA SWEEP — Pruned H384 (35.6M) + ADR-MSE, không Stage A")
print("=" * 84)
print(f"{'alpha':>6} {'w_KD':>6} {'w_CL':>6} | {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>9}")
print("-" * 84)

for a, r in sorted(alpha_results.items()):
    print(f"{a:>6.1f} {a:>6.1f} {1-a:>6.1f} | "
          f"{r['Hit@1']:>8.4f} {r['Recall@5']:>10.4f} "
          f"{r['MRR@10']:>9.4f} {r['NDCG@10']:>9.4f}")

print("-" * 84)

if alpha_results:
    best_a, best_r = max(alpha_results.items(), key=lambda x: x[1]["MRR@10"])
    print(f"Best theo MRR@10 : alpha = {best_a}  ({best_r['MRR@10']:.4f})")

    best_h1_a, best_h1_r = max(alpha_results.items(), key=lambda x: x[1]["Hit@1"])
    print(f"Best theo Hit@1  : alpha = {best_h1_a}  ({best_h1_r['Hit@1']:.4f})")

    # So α=1.0 (KD thuần) với α tốt nhất -> đo đóng góp của nhánh L_CL
    if 1.0 in alpha_results and best_a != 1.0:
        r1  = alpha_results[1.0]
        d_h1  = best_r["Hit@1"]  - r1["Hit@1"]
        d_mrr = best_r["MRR@10"] - r1["MRR@10"]
        print(f"\n>>> Đóng góp của nhánh L_CL (alpha={best_a} vs alpha=1.0):")
        print(f"    Hit@1  : {r1['Hit@1']:.4f} -> {best_r['Hit@1']:.4f}  ({d_h1:+.4f})")
        print(f"    MRR@10 : {r1['MRR@10']:.4f} -> {best_r['MRR@10']:.4f}  ({d_mrr:+.4f})")

# Lưu lại
with open(f"{SWEEP_DIR}/alpha_sweep_results.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in alpha_results.items()}, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: {SWEEP_DIR}/alpha_sweep_results.json")

gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Raw embedding shape: (1, 1024)
Encoding corpus...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Pruned ADR-MSE alpha=0.7:   0%|          | 0/390 [00:00<?, ?it/s]


Pruned ADR-MSE alpha=0.7
n:         390
Hit@1:    0.8282
Recall@5: 0.9590
MRR@10:   0.8844
NDCG@10:  0.9051
Saved: /content/drive/MyDrive/Data/archive/ablation/alpha_sweep_pruned_adrmse/rank_alpha07.jsonl

ALPHA = 1.0   (w_KD=1.0, w_CL=0.0)
/content/drive/MyDrive/Data/archive/ablation/alpha_sweep_pruned_adrmse/stage_b_adrmse_alpha10/best


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Pruned ADR-MSE alpha=1.0:   0%|          | 0/390 [00:00<?, ?it/s]


Pruned ADR-MSE alpha=1.0
n:         390
Hit@1:    0.8205
Recall@5: 0.9564
MRR@10:   0.8796
NDCG@10:  0.9015
Saved: /content/drive/MyDrive/Data/archive/ablation/alpha_sweep_pruned_adrmse/rank_alpha10.jsonl

ALPHA SWEEP — Pruned H384 (35.6M) + ADR-MSE, không Stage A
 alpha   w_KD   w_CL |    Hit@1   Recall@5    MRR@10   NDCG@10
------------------------------------------------------------------------------------
   0.3    0.3    0.7 |   0.8128     0.9692    0.8780    0.8997
   0.5    0.5    0.5 |   0.8205     0.9641    0.8815    0.9028
   0.7    0.7    0.3 |   0.8282     0.9590    0.8844    0.9051
   1.0    1.0    0.0 |   0.8205     0.9564    0.8796    0.9015
------------------------------------------------------------------------------------
Best theo MRR@10 : alpha = 0.7  (0.8844)
Best theo Hit@1  : alpha = 0.7  (0.8282)

>>> Đóng góp của nhánh L_CL (alpha=0.7 vs alpha=1.0):
    Hit@1  : 0.8205 -> 0.8282  (+0.0077)
    MRR@10 : 0.8796 -> 0.8844  (+0.0048)

Đã lưu: /content/drive/MyDriv

In [ ]:
# ===== VÁ LỖ HỔNG: H384 CÓ Stage A (để so với H384 KHÔNG Stage A) =====
import gc, torch
from pathlib import Path

H384_BASE    = MMARCO_BASE   # /content/drive/MyDrive/Data/archive/mmarco-mMiniLMv2-L12-H384-v1
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"

OUT_A = f"{ABL}/h384_with_stage_a/stage_a"
OUT_B = f"{ABL}/h384_with_stage_a/stage_b_adrmse_alpha07"

# ---- Stage A: BCE trên domain (giống hệt cấu hình MiniLM) ----
print("#" * 80)
print("STAGE A — H384")
print("#" * 80)

ckpt_h384_stage_a = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=None,
    dev_path=DOMAIN_DEV,
    output_dir=OUT_A,
    base_model=H384_BASE,
    domain_upsample=1,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    patience=2,
    seed=42,
)

gc.collect(); torch.cuda.empty_cache()

# ---- Stage B: ADR-MSE alpha=0.7 (giống hệt cấu hình H384 hiện tại) ----
print("\n" + "#" * 80)
print("STAGE B — H384 (sau Stage A)")
print("#" * 80)

ckpt_h384_a_plus_b = train_stage_b_ranknet(
    stage_a_checkpoint=ckpt_h384_stage_a,   # <-- KHÁC BIỆT DUY NHẤT so với cell 16
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=OUT_B,
    loss_type="adr_mse",
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
)

gc.collect(); torch.cuda.empty_cache()

print("\nDONE")
print("Stage A     :", ckpt_h384_stage_a)
print("Stage A + B :", ckpt_h384_a_plus_b)

################################################################################
STAGE A — H384
################################################################################


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Base: mmarco-mMiniLMv2-L12-H384-v1 | Seed: 42
Train: 2,328 | Dev: 305
Epoch 1/5 | Loss: 0.4017 | Dev Acc: 0.8623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8623)
Epoch 2/5 | Loss: 0.1054 | Dev Acc: 0.8721


...

  → Saved best (acc=0.8721)
Epoch 3/5 | Loss: 0.0427 | Dev Acc: 0.8525
Epoch 4/5 | Loss: 0.0164 | Dev Acc: 0.8459
Early stopping at epoch 4
Stage A done. Best dev acc: 0.8721

################################################################################
STAGE B — H384 (sau Stage A)
################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.3685 | KD(adr_mse): 0.4880 | CL: 0.0896 | Dev Acc: 0.8918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8918)


In [8]:
# ===== VÁ LỖ HỔNG: H384 CÓ Stage A (để so với H384 KHÔNG Stage A) =====
import gc, torch
from pathlib import Path
RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"
OUT_B = f"{ABL}/h384_with_stage_a/stage_b_adrmse_alpha07"

ckpt_h384_a_plus_b = train_stage_b_ranknet(
    stage_a_checkpoint="/content/drive/MyDrive/Data/archive/ablation/h384_with_stage_a/stage_a/best",
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=OUT_B,
    loss_type="adr_mse",
    epochs=5,
    batch_size=8,
    lr=1e-5,
    max_length=512,
    alpha=0.7,
    patience=2,
    seed=42,
)

gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.3685 | KD(adr_mse): 0.4881 | CL: 0.0895 | Dev Acc: 0.8918


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8918)
Epoch 2/5 | Loss: 0.3071 | KD(adr_mse): 0.4023 | CL: 0.0850 | Dev Acc: 0.9016


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.9016)
Epoch 3/5 | Loss: 0.2866 | KD(adr_mse): 0.3788 | CL: 0.0713 | Dev Acc: 0.9115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.9115)
Epoch 4/5 | Loss: 0.2777 | KD(adr_mse): 0.3673 | CL: 0.0686 | Dev Acc: 0.9082
Epoch 5/5 | Loss: 0.2689 | KD(adr_mse): 0.3576 | CL: 0.0620 | Dev Acc: 0.9115
Early stopping at epoch 5
adr_mse done. Best dev acc: 0.9115


In [11]:
# ===== BENCHMARK: H384 có Stage A (2 checkpoint) vs không Stage A =====
import gc, json, torch
from pathlib import Path

H384_STAGE_A_ONLY = f"{ABL}/h384_with_stage_a/stage_a/best"                    # chỉ Stage A
H384_A_PLUS_B     = f"{ABL}/h384_with_stage_a/stage_b_adrmse_alpha07/best"     # Stage A + B
H384_NO_A         = f"{ABL}/loss_ablation_h384_stage_b/stage_b_adrmse/best"    # đã có: 0.8862

# Nếu biến từ cell train còn trong RAM thì ưu tiên dùng
if "ckpt_h384_stage_a" in globals():
    H384_STAGE_A_ONLY = ckpt_h384_stage_a
if "ckpt_h384_a_plus_b" in globals():
    H384_A_PLUS_B = ckpt_h384_a_plus_b

if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64)

MODELS = [
    ("H384 chỉ Stage A",      H384_STAGE_A_ONLY),
    ("H384 Stage A + B",      H384_A_PLUS_B),
    ("H384 Stage B (không A)", H384_NO_A),
]

stage_a_ablation = {}

for name, path in MODELS:
    if not Path(path).exists():
        print(f"⚠️  SKIP missing: {name} -> {path}")
        continue

    print("\n" + "=" * 74)
    print(name)
    print(path)
    print("=" * 74)

    result, _ = benchmark_dense_then_rerank(
        reranker_path=path,
        name=name,
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=f"{ABL}/h384_with_stage_a/rank_{name.replace(' ', '_').replace('(', '').replace(')', '')}.jsonl",
    )

    stage_a_ablation[name] = result
    gc.collect()
    torch.cuda.empty_cache()


# ===== BẢNG TỔNG HỢP =====
print("\n" + "=" * 88)
print("ABLATION: Vai trò của Stage A trên H384 (ADR-MSE, alpha=0.7)")
print("=" * 88)
print(f"{'Cấu hình':<26} | {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>9}")
print("-" * 88)

for name, _ in MODELS:
    if name in stage_a_ablation:
        r = stage_a_ablation[name]
        print(f"{name:<26} | {r['Hit@1']:>8.4f} {r['Recall@5']:>10.4f} "
              f"{r['MRR@10']:>9.4f} {r['NDCG@10']:>9.4f}")

# So sánh cốt lõi: có A vs không A (cùng Stage B)
k_yes, k_no = "H384 Stage A + B", "H384 Stage B (không A)"
if k_yes in stage_a_ablation and k_no in stage_a_ablation:
    a, b = stage_a_ablation[k_yes], stage_a_ablation[k_no]
    n = len(dense_pool)
    print("-" * 88)
    print(f"{'Δ (có A − không A)':<26} | "
          f"{a['Hit@1']-b['Hit@1']:>+8.4f} {a['Recall@5']-b['Recall@5']:>+10.4f} "
          f"{a['MRR@10']-b['MRR@10']:>+9.4f} {a['NDCG@10']-b['NDCG@10']:>+9.4f}")

    d_mrr = a["MRR@10"] - b["MRR@10"]
    print(f"\n>>> ΔMRR@10 = {d_mrr:+.4f}  (n={n} → tương đương ~{abs(d_mrr)*n:.1f} query)")
    if abs(d_mrr) < 0.005:
        print(">>> KẾT LUẬN: Stage A KHÔNG đóng góp đáng kể trên H384 → xác nhận việc bỏ Stage A.")
    elif d_mrr > 0:
        print(">>> KẾT LUẬN: Stage A CÓ giúp trên H384 → cần cập nhật lại model cuối.")
    else:
        print(">>> KẾT LUẬN: Stage A làm GIẢM chất lượng → càng biện minh cho việc bỏ.")

with open(f"{ABL}/h384_with_stage_a/stage_a_ablation.json", "w", encoding="utf-8") as f:
    json.dump(stage_a_ablation, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: {ABL}/h384_with_stage_a/stage_a_ablation.json")

gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Raw embedding shape: (1, 1024)
Encoding corpus...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

H384 Stage A + B:   0%|          | 0/390 [00:00<?, ?it/s]


H384 Stage A + B
n:         390
Hit@1:    0.8256
Recall@5: 0.9615
MRR@10:   0.8837
NDCG@10:  0.9050
Saved: /content/drive/MyDrive/Data/archive/ablation/h384_with_stage_a/rank_H384_Stage_A_+_B.jsonl

H384 Stage B (không A)
/content/drive/MyDrive/Data/archive/ablation/loss_ablation_h384_stage_b/stage_b_adrmse/best


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

H384 Stage B (không A):   0%|          | 0/390 [00:00<?, ?it/s]


H384 Stage B (không A)
n:         390
Hit@1:    0.8308
Recall@5: 0.9615
MRR@10:   0.8862
NDCG@10:  0.9069
Saved: /content/drive/MyDrive/Data/archive/ablation/h384_with_stage_a/rank_H384_Stage_B_không_A.jsonl

ABLATION: Vai trò của Stage A trên H384 (ADR-MSE, alpha=0.7)
Cấu hình                   |    Hit@1   Recall@5    MRR@10   NDCG@10
----------------------------------------------------------------------------------------
H384 chỉ Stage A           |   0.8051     0.9462    0.8646    0.8847
H384 Stage A + B           |   0.8256     0.9615    0.8837    0.9050
H384 Stage B (không A)     |   0.8308     0.9615    0.8862    0.9069
----------------------------------------------------------------------------------------
Δ (có A − không A)         |  -0.0051    +0.0000   -0.0025   -0.0018

>>> ΔMRR@10 = -0.0025  (n=390 → tương đương ~1.0 query)
>>> KẾT LUẬN: Stage A KHÔNG đóng góp đáng kể trên H384 → xác nhận việc bỏ Stage A.

Đã lưu: /content/drive/MyDrive/Data/archive/ablation/h384_with_st

In [12]:
# ===== CHẨN ĐOÁN TOKENIZER: tại sao MiniLM thất bại =====
from transformers import AutoTokenizer
import numpy as np

SAMPLES = [
    "Nhiệt độ vận hành tối đa của máy biến áp là bao nhiêu?",
    "Tính chi phí nhiên liệu cho tổ máy phát điện",
    "Hằng số điện môi trong chân không có giá trị bao nhiêu?",
]

for name, path in [("MiniLM (Anh)", MINILM_BASE), ("H384 (đa ngữ)", MMARCO_BASE)]:
    tok = AutoTokenizer.from_pretrained(path)
    print("\n" + "=" * 72)
    print(f"{name}  |  vocab = {tok.vocab_size:,}")
    print("=" * 72)

    ferts = []
    for s in SAMPLES:
        ids    = tok(s, add_special_tokens=False)["input_ids"]
        toks   = tok.convert_ids_to_tokens(ids)
        decoded = tok.decode(ids)
        n_words = len(s.split())
        ferts.append(len(ids) / n_words)

        print(f"\nGốc     : {s}")
        print(f"Tokens  : {toks[:22]}{' ...' if len(toks) > 22 else ''}")
        print(f"Decode  : {decoded}")          # ⚠️ NHÌN KỸ DÒNG NÀY — còn dấu không?
        print(f"→ {n_words} từ → {len(ids)} token  (fertility {len(ids)/n_words:.2f})")

    print(f"\n>>> Fertility trung bình: {np.mean(ferts):.2f} token/từ")

    # Kiểm tra trực tiếp xem dấu có bị xoá không
    probe = tok.convert_ids_to_tokens(tok("điện", add_special_tokens=False)["input_ids"])
    print(f">>> 'điện' → {probe}   {'⚠️ MẤT DẤU!' if 'đ' not in ''.join(probe) else '✓ giữ dấu'}")


MiniLM (Anh)  |  vocab = 30,522

Gốc     : Nhiệt độ vận hành tối đa của máy biến áp là bao nhiêu?
Tokens  : ['nh', '##ie', '##t', 'đ', '##o', 'van', 'han', '##h', 'to', '##i', 'đ', '##a', 'cu', '##a', 'may', 'bien', 'ap', 'la', 'bao', 'nh', '##ieu', '?']
Decode  : nhiet đo van hanh toi đa cua may bien ap la bao nhieu?
→ 13 từ → 22 token  (fertility 1.69)

Gốc     : Tính chi phí nhiên liệu cho tổ máy phát điện
Tokens  : ['tin', '##h', 'chi', 'phi', 'nh', '##ien', 'lieu', 'cho', 'to', 'may', 'ph', '##at', 'đ', '##ien']
Decode  : tinh chi phi nhien lieu cho to may phat đien
→ 10 từ → 14 token  (fertility 1.40)

Gốc     : Hằng số điện môi trong chân không có giá trị bao nhiêu?
Tokens  : ['hang', 'so', 'đ', '##ien', 'moi', 'tr', '##ong', 'chan', 'k', '##hong', 'co', 'gia', 'tri', 'bao', 'nh', '##ieu', '?']
Decode  : hang so đien moi trong chan khong co gia tri bao nhieu?
→ 12 từ → 17 token  (fertility 1.42)

>>> Fertility trung bình: 1.50 token/từ
>>> 'điện' → ['đ', '##ien']   ✓ giữ dấu

H

In [13]:
# ===== ĐỊNH LƯỢNG: sụp đổ đồng âm do mất dấu =====
import unicodedata
from collections import defaultdict

def strip_accents(s):
    """Mô phỏng đúng thứ BERT uncased làm: NFD rồi bỏ dấu tổ hợp (Mn)."""
    return "".join(c for c in unicodedata.normalize("NFD", s.lower())
                   if unicodedata.category(c) != "Mn")

# Gom từ vựng từ corpus + query
vocab_words = set()
for t in texts:
    vocab_words.update(w.strip(".,;:!?()[]\"'") for w in t.split())
for q in test_queries:
    vocab_words.update(w.strip(".,;:!?()[]\"'") for w in q["question"].split())
vocab_words = {w for w in vocab_words if w and any(c.isalpha() for c in w)}

# Nhóm các từ bị gộp thành cùng một chuỗi sau khi lột dấu
collapse = defaultdict(set)
for w in vocab_words:
    collapse[strip_accents(w)].add(w)

collided = {k: v for k, v in collapse.items() if len(v) > 1}
n_words_lost = sum(len(v) for v in collided.values())

print(f"Tổng từ khác nhau trong corpus : {len(vocab_words):,}")
print(f"Số chuỗi sau khi lột dấu       : {len(collapse):,}")
print(f"→ Mất {len(vocab_words) - len(collapse):,} từ phân biệt "
      f"({(1 - len(collapse)/len(vocab_words))*100:.1f}% vốn từ bị gộp)")
print(f"→ {n_words_lost:,} từ ({n_words_lost/len(vocab_words)*100:.1f}%) "
      f"rơi vào một nhóm đồng âm giả\n")

print("20 nhóm sụp đổ nặng nhất:")
for k, v in sorted(collided.items(), key=lambda x: -len(x[1]))[:20]:
    print(f"  {k:<14} ← {sorted(v)}")

Tổng từ khác nhau trong corpus : 7,165
Số chuỗi sau khi lột dấu       : 4,531
→ Mất 2,634 từ phân biệt (36.8% vốn từ bị gộp)
→ 3,640 từ (50.8%) rơi vào một nhóm đồng âm giả

20 nhóm sụp đổ nặng nhất:
  van            ← ['Van', 'Vân', 'Văn', 'Vấn', 'Vẩn', 'Vẫn', 'Vận', 'Vặn', 'van', 'ván', 'văn', 'vạn', 'vấn', 'vẩn', 'vẫn', 'vận', 'vặn']
  đo             ← ['Đo', 'Đó', 'Đô', 'ĐỒ', 'Đồ', 'Đỗ', 'Độ', 'Đỡ', 'đo', 'đó', 'đô', 'đỏ', 'đồ', 'đổ', 'đỗ', 'độ', 'đỡ']
  co             ← ['CO', 'Có', 'CƠ', 'Cơ', 'CỐ', 'Cổ', 'Cờ', 'Cở', 'co', 'có', 'cô', 'cơ', 'cố', 'cổ', 'cỡ', 'cợ']
  ho             ← ['Ho', 'Họ', 'Hố', 'Hồ', 'Hổ', 'Hỗ', 'Hộ', 'ho', 'họ', 'hố', 'hồ', 'hổ', 'hỗ', 'hộ', 'hở']
  can            ← ['Cán', 'Cân', 'Căn', 'CẦN', 'Cần', 'Cặn', 'can', 'cán', 'cân', 'căn', 'cạn', 'cản', 'cần', 'cận', 'cặn']
  dung           ← ['DUNG', 'Dung', 'Dùng', 'Dũng', 'DỤNG', 'Dụng', 'Dừng', 'DỰNG', 'Dựng', 'dung', 'dùng', 'dưng', 'dụng', 'dừng', 'dựng']
  an             ← ['AN', 'An', 'an', 'ÁN', 'án'

In [6]:
# ===== CẮT TẦNG: L12 → L6 (giữ kiến thức mMARCO) =====
import torch, gc, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"
SRC   = "/content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_base_same_old"   # base đã pruned vocab (35.6M)
L6_BASE = f"{ABL}/mMiniLM_H384_pruned_L6_base"
KEEP  = [0, 2, 4, 6, 8, 10]     # mỗi lớp cách một — công thức DistilBERT

model = AutoModelForSequenceClassification.from_pretrained(SRC)
tok   = AutoTokenizer.from_pretrained(SRC)

n_before = sum(p.numel() for p in model.parameters())
layers   = model.roberta.encoder.layer
print(f"Trước: {len(layers)} lớp | {n_before/1e6:.1f}M params")

# Giữ lại 6 lớp
model.roberta.encoder.layer = torch.nn.ModuleList([layers[i] for i in KEEP])
model.config.num_hidden_layers = len(KEEP)

n_after = sum(p.numel() for p in model.parameters())
print(f"Sau  : {len(KEEP)} lớp | {n_after/1e6:.1f}M params "
      f"(giảm {(1-n_after/n_before)*100:.1f}%)")

os.makedirs(L6_BASE, exist_ok=True)
model.save_pretrained(L6_BASE)
tok.save_pretrained(L6_BASE)
print("Saved:", L6_BASE)

del model; gc.collect(); torch.cuda.empty_cache()

# ===== Stage B trên L6 =====
ckpt_l6 = train_stage_b_ranknet(
    stage_a_checkpoint=L6_BASE,
    rerank_path=RERANK_LOGIT,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=f"{ABL}/stage_b_pruned_L6_adrmse_alpha07",
    loss_type="adr_mse",
    epochs=5, batch_size=8, lr=1e-5, max_length=512,
    alpha=0.7, patience=2, seed=42,
)
print("DONE:", ckpt_l6)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trước: 12 lớp | 35.6M params
Sau  : 6 lớp | 24.9M params (giảm 29.9%)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Data/archive/ablation/mMiniLM_H384_pruned_L6_base


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

  → Saved best (acc=0.7311)
Epoch 4/5 | Loss: 0.6828 | KD(adr_mse): 0.7517 | CL: 0.5221 | Dev Acc: 0.7639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7639)
Epoch 5/5 | Loss: 0.6549 | KD(adr_mse): 0.7135 | CL: 0.5183 | Dev Acc: 0.7836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.7836)
adr_mse done. Best dev acc: 0.7836
DONE: /content/drive/MyDrive/Data/archive/ablation/stage_b_pruned_L6_adrmse_alpha07/best


In [9]:
# ===== BENCHMARK L6: chất lượng + tốc độ, so với L12 =====
import gc, time, json
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

L6_CKPT  = f"{ABL}/stage_b_pruned_L6_adrmse_alpha07/best"
L12_CKPT = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"   # Pruned L12, đã có 0.8844

if "ckpt_l6" in globals():
    L6_CKPT = ckpt_l6

if "dense_pool" not in globals():
    dense_pool, _ = build_dense_topk_pool_512(
        embed_model_path=EMB_FT, top_k=TOP_K_DENSE, dim=EMB_DIM, batch_size=64)

sample   = dense_pool[0]
q_sample = sample["question"]
d_sample = sample["candidate_texts"][:20]


@torch.no_grad()
def measure(path, n_warmup=3, n_repeat=30, max_length=512):
    tok   = AutoTokenizer.from_pretrained(path, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters())
    size_mb  = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
    n_layers = model.config.num_hidden_layers

    def _pass():
        enc = tok([q_sample]*len(d_sample), d_sample, padding=True, truncation=True,
                  max_length=max_length, return_tensors="pt").to(device)
        return model(**enc).logits

    for _ in range(n_warmup):
        _pass()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        _pass()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    del model, tok
    gc.collect(); torch.cuda.empty_cache()

    return {
        "layers":   n_layers,
        "params_M": n_params / 1e6,
        "size_MB":  size_mb,
        "ms_per_q": float(np.median(times)),
        "p95_ms":   float(np.percentile(times, 95)),
    }


rows = {}

for name, path in [("Pruned L6", L6_CKPT), ("Pruned L12", L12_CKPT)]:
    if not Path(path).exists():
        print(f"⚠️  SKIP missing: {name} -> {path}")
        continue

    print("\n" + "=" * 74)
    print(f"{name}\n{path}")
    print("=" * 74)

    speed = measure(path)
    print(f"  Layers   : {speed['layers']}")
    print(f"  Params   : {speed['params_M']:.1f} M")
    print(f"  Size     : {speed['size_MB']:.1f} MB")
    print(f"  Rerank@20: {speed['ms_per_q']:.1f} ms/query (p95 {speed['p95_ms']:.1f})")

    quality, _ = benchmark_dense_then_rerank(
        reranker_path=path,
        name=name,
        dense_pool=dense_pool,
        batch_size=32,
        max_length=512,
        save_rank_path=f"{ABL}/rank_{name.replace(' ', '_').lower()}.jsonl",
    )

    rows[name] = {**speed, **quality}
    gc.collect(); torch.cuda.empty_cache()


# ===== BẢNG PARETO =====
TEACHER_MS, TEACHER_MRR = 385.5, 0.8862

print("\n" + "=" * 100)
print("PARETO: Layer pruning (L12 → L6)")
print("=" * 100)
print(f"{'Model':<16} {'L':>3} {'Params':>9} {'Size':>9} {'ms/q':>8} "
      f"{'Hit@1':>8} {'MRR@10':>9} {'NDCG@10':>9} {'Speedup':>9}")
print("-" * 100)

for name in ["Pruned L12", "Pruned L6"]:
    if name not in rows:
        continue
    r = rows[name]
    print(f"{name:<16} {r['layers']:>3} {r['params_M']:>8.1f}M {r['size_MB']:>7.1f}MB "
          f"{r['ms_per_q']:>8.1f} {r['Hit@1']:>8.4f} {r['MRR@10']:>9.4f} "
          f"{r['NDCG@10']:>9.4f} {TEACHER_MS/r['ms_per_q']:>8.1f}x")

print(f"{'BGE teacher':<16} {24:>3} {567.8:>8.1f}M {2187.0:>7.1f}MB "
      f"{TEACHER_MS:>8.1f} {0.8256:>8.4f} {TEACHER_MRR:>9.4f} {0.9068:>9.4f} {1.0:>8.1f}x")
print(f"{'ViRanker':<16} {24:>3} {567.8:>8.1f}M {2165.8:>7.1f}MB "
      f"{385.7:>8.1f} {0.7872:>8.4f} {0.8593:>9.4f} {0.8844:>9.4f} {1.0:>8.1f}x")

# ===== Trade-off L6 vs L12 =====
if "Pruned L6" in rows and "Pruned L12" in rows:
    l6, l12 = rows["Pruned L6"], rows["Pruned L12"]
    d_mrr = l6["MRR@10"] - l12["MRR@10"]
    speedup = l12["ms_per_q"] / l6["ms_per_q"]

    print("\n" + "-" * 100)
    print(f">>> L6 vs L12: nhanh hơn {speedup:.2f}x, ΔMRR@10 = {d_mrr:+.4f} "
          f"(~{abs(d_mrr)*len(dense_pool):.1f} query)")
    print(f">>> So ViRanker (0.8593 @ 385.7ms): L6 "
          f"{'VƯỢT' if l6['MRR@10'] > 0.8593 else 'THUA'} "
          f"({l6['MRR@10']:.4f}) và nhanh hơn {385.7/l6['ms_per_q']:.1f}x")
    print(f">>> So không-rerank (0.7929): L6 {l6['MRR@10'] - 0.7929:+.4f}")

with open(f"{ABL}/l6_vs_l12.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: {ABL}/l6_vs_l12.json")

gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Raw embedding shape: (1, 1024)
Encoding corpus...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Layers   : 12
  Params   : 35.6 M
  Size     : 135.8 MB
  Rerank@20: 50.4 ms/query (p95 53.3)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Pruned L12:   0%|          | 0/390 [00:00<?, ?it/s]


Pruned L12
n:         390
Hit@1:    0.8282
Recall@5: 0.9590
MRR@10:   0.8844
NDCG@10:  0.9051
Saved: /content/drive/MyDrive/Data/archive/ablation/rank_pruned_l12.jsonl

PARETO: Layer pruning (L12 → L6)
Model              L    Params      Size     ms/q    Hit@1    MRR@10   NDCG@10   Speedup
----------------------------------------------------------------------------------------------------
Pruned L12        12     35.6M   135.8MB     50.4   0.8282    0.8844    0.9051      7.7x
Pruned L6          6     24.9M    95.2MB     30.2   0.5769    0.7161    0.7709     12.8x
BGE teacher       24    567.8M  2187.0MB    385.5   0.8256    0.8862    0.9068      1.0x
ViRanker          24    567.8M  2165.8MB    385.7   0.7872    0.8593    0.8844      1.0x

----------------------------------------------------------------------------------------------------
>>> L6 vs L12: nhanh hơn 1.67x, ΔMRR@10 = -0.1683 (~65.6 query)
>>> So ViRanker (0.8593 @ 385.7ms): L6 THUA (0.7161) và nhanh hơn 12.8x
>>> So không-

In [9]:
import torch, gc, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT   = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"
MAX_Q  = 32
MAX_D  = 480

tok   = AutoTokenizer.from_pretrained(CKPT, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, attn_implementation="eager"          # ← FIX
).to(device).eval()

PAD = model.config.pad_token_id
BOS = tok.bos_token_id
EOS = tok.eos_token_id
N_LAYERS   = model.config.num_hidden_layers
POS_OFFSET = PAD + 1

print(f"layers={N_LAYERS} | pad={PAD} bos={BOS} eos={EOS}")
print(f"attn_impl = {model.config._attn_implementation}")
print(f"max_position_emb = {model.config.max_position_embeddings}")
print(f"→ position cao nhất: {POS_OFFSET + MAX_Q + MAX_D - 1}")
assert POS_OFFSET + MAX_Q + MAX_D <= model.config.max_position_embeddings, "TRÀN position!"

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

layers=12 | pad=1 bos=0 eos=2
attn_impl = eager
max_position_emb = 514
→ position cao nhất: 513


In [11]:
# ===== PreTTR (bản robust — chỉ dùng public API) =====
import contextlib
import torch, gc, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CKPT  = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"
MAX_Q, MAX_D = 32, 480

tok   = AutoTokenizer.from_pretrained(CKPT, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).to(device).eval()
# ↑ BỎ attn_implementation="eager" — giờ không cần, và sdpa nhanh hơn

PAD, BOS, EOS = model.config.pad_token_id, tok.bos_token_id, tok.eos_token_id
N_LAYERS   = model.config.num_hidden_layers
POS_OFFSET = PAD + 1

Q_POS = torch.arange(POS_OFFSET, POS_OFFSET + MAX_Q)
D_POS = torch.arange(POS_OFFSET + MAX_Q, POS_OFFSET + MAX_Q + MAX_D)

print(f"layers={N_LAYERS} | attn={model.config._attn_implementation}")


def build_q(q):
    ids  = [BOS] + tok(q, add_special_tokens=False)["input_ids"][:MAX_Q-2] + [EOS]
    mask = [1]*len(ids) + [0]*(MAX_Q-len(ids))
    return ids + [PAD]*(MAX_Q-len(ids)), mask

def build_d(d):
    ids  = [EOS] + tok(d, add_special_tokens=False)["input_ids"][:MAX_D-2] + [EOS]
    mask = [1]*len(ids) + [0]*(MAX_D-len(ids))
    return ids + [PAD]*(MAX_D-len(ids)), mask


class _PassThrough(torch.nn.Module):
    """Giả làm embeddings: trả thẳng hidden state đã có."""
    def __init__(self, h):
        super().__init__()
        self._h = h
    def forward(self, *a, **k):
        return self._h


@contextlib.contextmanager
def _sliced(hidden, lo, hi):
    rob = model.roberta
    emb0, lay0 = rob.embeddings, rob.encoder.layer
    rob.embeddings   = _PassThrough(hidden)
    rob.encoder.layer = torch.nn.ModuleList(list(lay0)[lo:hi])
    try:
        yield
    finally:
        rob.embeddings   = emb0
        rob.encoder.layer = lay0


def run_layers(hidden, mask, lo, hi):
    """Chạy layer [lo, hi) qua public API -> mask do thư viện tự lo."""
    if lo >= hi:
        return hidden
    with _sliced(hidden, lo, hi):
        out = model.roberta(inputs_embeds=hidden, attention_mask=mask)
    return out.last_hidden_state


def embed(ids, pos):
    return model.roberta.embeddings(input_ids=ids, position_ids=pos)


@torch.no_grad()
def precompute_docs(docs, split_layer, batch_size=32):
    """PHA 1 (offline): doc qua layer 0..l, KHÔNG thấy query."""
    hs, ms = [], []
    for i in range(0, len(docs), batch_size):
        built = [build_d(d) for d in docs[i:i+batch_size]]
        ids  = torch.tensor([b[0] for b in built], device=device)
        mask = torch.tensor([b[1] for b in built], device=device)
        pos  = D_POS.unsqueeze(0).expand(len(built), -1).to(device)

        h = run_layers(embed(ids, pos), mask, 0, split_layer)
        hs.append(h.cpu()); ms.append(mask.cpu())
    return torch.cat(hs), torch.cat(ms)


@torch.no_grad()
def score_pretr(query, doc_h, doc_m, split_layer):
    """PHA 2: query qua 0..l -> ghép -> cross-attention l..L."""
    n = doc_h.size(0)
    qi, qm = build_q(query)
    q_ids  = torch.tensor([qi], device=device).expand(n, -1)
    q_mask = torch.tensor([qm], device=device).expand(n, -1)
    q_pos  = Q_POS.unsqueeze(0).expand(n, -1).to(device)

    qh = run_layers(embed(q_ids, q_pos), q_mask, 0, split_layer)

    hidden = torch.cat([qh, doc_h.to(device)], dim=1)
    mask   = torch.cat([q_mask, doc_m.to(device)], dim=1)
    hidden = run_layers(hidden, mask, split_layer, N_LAYERS)
    return model.classifier(hidden).squeeze(-1)


@torch.no_grad()
def score_full(query, docs):
    """Baseline: full model, CÙNG token + position."""
    qi, qm = build_q(query)
    built  = [build_d(d) for d in docs]
    ids  = torch.tensor([qi + b[0] for b in built], device=device)
    mask = torch.tensor([qm + b[1] for b in built], device=device)
    pos  = torch.cat([Q_POS, D_POS]).unsqueeze(0).expand(len(docs), -1).to(device)
    return model(input_ids=ids, attention_mask=mask, position_ids=pos).logits.squeeze(-1)


# ===== KIỂM CHỨNG =====
sample = dense_pool[0]
q_test, d_test = sample["question"], sample["candidate_texts"][:5]

full   = score_full(q_test, d_test)
dh, dm = precompute_docs(d_test, split_layer=0)
split0 = score_pretr(q_test, dh, dm, split_layer=0)

diff = (full - split0).abs().max().item()
print("\nfull   :", full.cpu().numpy().round(4))
print("split@0:", split0.cpu().numpy().round(4))
print(f"\nmax diff = {diff:.6f}")
print("✓ KHỚP — đi tiếp được" if diff < 1e-3 else "✗ SAI — dừng, debug")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

layers=12 | attn=sdpa

full   : [ 6.158  -5.142  -6.1934 -3.2665 -8.8792]
split@0: [ 6.158  -5.142  -6.1934 -3.2665 -8.8792]

max diff = 0.000000
✓ KHỚP — đi tiếp được


In [12]:
# ===== PreTTR: quét split_layer =====
import time, gc, json
import numpy as np
import torch
from tqdm.auto import tqdm

CACHE_FP16 = True     # nửa bộ nhớ, sai số không đáng kể
CACHE_GPU  = True     # giữ cache trên GPU (corpus nhỏ) — tránh tính CPU→GPU vào latency

cid2idx = {c: i for i, c in enumerate(cids)}
print(f"Corpus: {len(texts)} chunks | Queries: {len(dense_pool)}")

est_mb = len(texts) * MAX_D * model.config.hidden_size * (2 if CACHE_FP16 else 4) / 1024**2
print(f"Cache ước tính: {est_mb:.0f} MB ({'fp16' if CACHE_FP16 else 'fp32'})\n")


@torch.no_grad()
def eval_pretr(split_layer, batch_size=32, n_warmup=3):
    # ---- PHA 1: precompute TOÀN BỘ corpus (offline, làm 1 lần lúc index) ----
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    all_h, all_m = precompute_docs(texts, split_layer, batch_size=batch_size)
    torch.cuda.synchronize()
    t_pre = time.perf_counter() - t0

    if CACHE_FP16:
        all_h = all_h.half()
    if CACHE_GPU:
        all_h, all_m = all_h.to(device), all_m.to(device)

    cache_mb = all_h.numel() * all_h.element_size() / 1024**2

    # ---- warmup ----
    warm = dense_pool[0]
    widx = [cid2idx[c] for c in warm["candidate_ids"]]
    for _ in range(n_warmup):
        score_pretr(warm["question"], all_h[widx].float(), all_m[widx], split_layer)
    torch.cuda.synchronize()

    # ---- PHA 2: query-time ----
    metrics, times = [], []
    for rec in tqdm(dense_pool, desc=f"l={split_layer}", leave=False):
        idx = [cid2idx[c] for c in rec["candidate_ids"]]

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        scores = score_pretr(rec["question"], all_h[idx].float(), all_m[idx], split_layer)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

        order  = torch.argsort(-scores).cpu().numpy()
        ranked = [rec["candidate_ids"][i] for i in order]
        metrics.append(_metrics_one(ranked, rec["gold"]))

    r = summarize_metrics(metrics, f"PreTTR l={split_layer}", len(metrics))
    r.update({
        "split_layer":  split_layer,
        "ms_per_query": float(np.median(times)),
        "p95_ms":       float(np.percentile(times, 95)),
        "precompute_s": t_pre,
        "cache_MB":     cache_mb,
    })

    del all_h, all_m
    gc.collect(); torch.cuda.empty_cache()
    return r


results_pretr = {}
for l in [0, 6, 8, 10, 11]:
    r = eval_pretr(l)
    results_pretr[l] = r
    print(f"l={l:>2} | MRR@10 {r['MRR@10']:.4f} | Hit@1 {r['Hit@1']:.4f} "
          f"| {r['ms_per_query']:6.1f} ms | cache {r['cache_MB']:.0f}MB "
          f"| precompute {r['precompute_s']:.1f}s")


# ===== BẢNG =====
base = results_pretr[0]
print("\n" + "=" * 94)
print("PreTTR — quét split_layer  (l=0 là baseline: không tính trước gì)")
print("=" * 94)
print(f"{'l':>3} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>9} {'NDCG@10':>9} "
      f"{'ms/query':>10} {'Speedup':>9} {'ΔMRR':>9} {'Cache':>8}")
print("-" * 94)

for l, r in sorted(results_pretr.items()):
    print(f"{l:>3} {r['Hit@1']:>8.4f} {r['Recall@5']:>10.4f} {r['MRR@10']:>9.4f} "
          f"{r['NDCG@10']:>9.4f} {r['ms_per_query']:>10.1f} "
          f"{base['ms_per_query']/r['ms_per_query']:>8.1f}x "
          f"{r['MRR@10']-base['MRR@10']:>+9.4f} {r['cache_MB']:>7.0f}M")

print("-" * 94)
print(f"Tham chiếu: BGE teacher 0.8862 @ 385.5ms | ViRanker 0.8593 @ 385.7ms "
      f"| Không rerank 0.7929")

with open(f"{ABL}/pretr_sweep.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in results_pretr.items()}, f, indent=2, ensure_ascii=False)
print(f"\nĐã lưu: {ABL}/pretr_sweep.json")

gc.collect(); torch.cuda.empty_cache()

Corpus: 813 chunks | Queries: 390
Cache ước tính: 286 MB (fp16)



l=0:   0%|          | 0/390 [00:00<?, ?it/s]

l= 0 | MRR@10 0.8843 | Hit@1 0.8282 |   40.4 ms | cache 286MB | precompute 1.2s


l=6:   0%|          | 0/390 [00:00<?, ?it/s]

l= 6 | MRR@10 0.4976 | Hit@1 0.3564 |   25.5 ms | cache 286MB | precompute 1.9s


...

l= 8 | MRR@10 0.4074 | Hit@1 0.2692 |   20.3 ms | cache 286MB | precompute 2.2s


l=10:   0%|          | 0/390 [00:00<?, ?it/s]

l=10 | MRR@10 0.1727 | Hit@1 0.0641 |   15.2 ms | cache 286MB | precompute 2.4s


l=11:   0%|          | 0/390 [00:00<?, ?it/s]

l=11 | MRR@10 0.2050 | Hit@1 0.0846 |   12.5 ms | cache 286MB | precompute 2.5s

PreTTR — quét split_layer  (l=0 là baseline: không tính trước gì)
  l    Hit@1   Recall@5    MRR@10   NDCG@10   ms/query   Speedup      ΔMRR    Cache
----------------------------------------------------------------------------------------------
  0   0.8282     0.9641    0.8843    0.9054       40.4      1.0x   +0.0000     286M
  6   0.3564     0.6692    0.4976    0.5761       25.5      1.6x   -0.3867     286M
  8   0.2692     0.5846    0.4074    0.4928       20.3      2.0x   -0.4769     286M
 10   0.0641     0.2974    0.1727    0.2602       15.2      2.7x   -0.7116     286M
 11   0.0846     0.3359    0.2050    0.2924       12.5      3.2x   -0.6793     286M
----------------------------------------------------------------------------------------------
Tham chiếu: BGE teacher 0.8862 @ 385.5ms | ViRanker 0.8593 @ 385.7ms | Không rerank 0.7929

Đã lưu: /content/drive/MyDrive/Data/archive/ablation/pretr_sweep.js

In [14]:
# ===== TRAIN PreTTR: model học kiến trúc tách =====
import random, json, gc
import numpy as np
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm


RERANK_LOGIT = "/content/drive/MyDrive/Data/archive/retrieve_rerank_991_logit.jsonl"
SPLIT_LAYER = 6          # thử 6 trước; nếu ổn thì thử 8
ALPHA       = 0.7
EPOCHS      = 5
LR          = 1e-5
ACCUM       = 4          # gradient accumulation (mỗi query = 20 seq, nặng)

OUT = f"{ABL}/pretr_l{SPLIT_LAYER}_adrmse"
Path(OUT).mkdir(parents=True, exist_ok=True)


def score_pretr_grad(query, docs, split_layer):
    """Giống score_pretr nhưng CÓ gradient — doc tính on-the-fly."""
    n = len(docs)
    built  = [build_d(d) for d in docs]
    d_ids  = torch.tensor([b[0] for b in built], device=device)
    d_mask = torch.tensor([b[1] for b in built], device=device)
    d_pos  = D_POS.unsqueeze(0).expand(n, -1).to(device)
    dh = run_layers(embed(d_ids, d_pos), d_mask, 0, split_layer)   # doc KHÔNG thấy query

    qi, qm = build_q(query)
    q_ids  = torch.tensor([qi], device=device).expand(n, -1)
    q_mask = torch.tensor([qm], device=device).expand(n, -1)
    q_pos  = Q_POS.unsqueeze(0).expand(n, -1).to(device)
    qh = run_layers(embed(q_ids, q_pos), q_mask, 0, split_layer)   # query KHÔNG thấy doc

    hidden = torch.cat([qh, dh], dim=1)
    mask   = torch.cat([q_mask, d_mask], dim=1)
    hidden = run_layers(hidden, mask, split_layer, N_LAYERS)       # cross-attention từ đây
    return model.classifier(hidden).squeeze(-1)


# ---- data ----
kd_recs = [json.loads(l) for l in open(RERANK_LOGIT, encoding="utf-8") if l.strip()]
cl_recs = [json.loads(l) for l in open(DOMAIN_TRAIN, encoding="utf-8") if l.strip()]
print(f"KD: {len(kd_recs)} | CL: {len(cl_recs)} | split_layer={SPLIT_LAYER}")

kd_crit = ADRMSELoss()
bce     = torch.nn.BCEWithLogitsLoss()

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(kd_recs) // ACCUM) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.05*total_steps), total_steps)

rng = random.Random(42)
best_mrr = 0.0

for epoch in range(EPOCHS):
    model.train()
    rng.shuffle(kd_recs)
    tot = 0.0
    optimizer.zero_grad()

    for step, rec in enumerate(tqdm(kd_recs, desc=f"epoch {epoch+1}")):
        cands = rec["candidates"][:20]
        docs  = [c["chunk"] for c in cands]
        ranks = torch.tensor([c.get("rank", 0) for c in cands],
                             dtype=torch.float, device=device).unsqueeze(0)

        logits  = score_pretr_grad(rec["question"], docs, SPLIT_LAYER).unsqueeze(0)
        kd_loss = kd_crit(logits, ranks)

        # nhánh CL (song song, giữ domain)
        cl_loss = torch.tensor(0.0, device=device)
        if ALPHA < 1.0:
            d = rng.choice(cl_recs)
            pn = score_pretr_grad(d["query"], [d["positive"], d["negative"]], SPLIT_LAYER)
            cl_loss = (bce(pn[0:1], torch.ones(1, device=device)) +
                       bce(pn[1:2], torch.zeros(1, device=device))) / 2

        loss = (ALPHA * kd_loss + (1 - ALPHA) * cl_loss) / ACCUM
        loss.backward()
        tot += loss.item() * ACCUM

        if (step + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

    # ---- eval bằng chính MRR (không phải pairwise acc) ----
    model.eval()
    r = eval_pretr(SPLIT_LAYER)
    print(f"Epoch {epoch+1} | loss {tot/len(kd_recs):.4f} | "
          f"MRR@10 {r['MRR@10']:.4f} | Hit@1 {r['Hit@1']:.4f}")

    if r["MRR@10"] > best_mrr:
        best_mrr = r["MRR@10"]
        model.save_pretrained(f"{OUT}/best"); tok.save_pretrained(f"{OUT}/best")
        print(f"  → saved best (MRR {best_mrr:.4f})")

print(f"\nDONE | best MRR@10 = {best_mrr:.4f}  (baseline l=0: 0.8843)")

KD: 991 | CL: 2328 | split_layer=6


epoch 1:   0%|          | 0/991 [00:00<?, ?it/s]

l=6:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 1 | loss 0.6751 | MRR@10 0.8173 | Hit@1 0.7231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 4 | loss 0.3524 | MRR@10 0.8563 | Hit@1 0.7897


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → saved best (MRR 0.8563)


epoch 5:   0%|          | 0/991 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
!pip install -q -U "optimum[onnxruntime]>=1.24" onnx onnxruntime

In [7]:
print(f"dense_pool: {len(dense_pool)} queries | corpus: {len(cids)} chunks")
print(f"Dense trước rerank MRR@10: {res_dense['MRR@10']:.4f}")
ABL = "/content/drive/MyDrive/Data/archive/ablation"
BEST_CKPT = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"

dense_pool: 390 queries | corpus: 813 chunks
Dense trước rerank MRR@10: 0.7943


In [9]:
# ===== KIỂM TRA FILE + EXPORT ONNX + INT8 =====
import gc, os
from pathlib import Path
import numpy as np
from transformers import AutoTokenizer, AutoConfig

ABL       = "/content/drive/MyDrive/Data/archive/ablation"
BEST_CKPT = f"{ABL}/stage_b_pruned_h384_adr_mse_alpha07/best"
ONNX_FP32 = "/content/onnx_fp32"
ONNX_INT8 = "/content/onnx_int8"

print("Files trong checkpoint:")
for f in sorted(Path(BEST_CKPT).iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size/1024**2:8.2f} MB")

# ---- CHỈ dùng fast tokenizer (tokenizer.json là bản đã prune) ----
tok = AutoTokenizer.from_pretrained(BEST_CKPT)          # mặc định use_fast=True
cfg = AutoConfig.from_pretrained(BEST_CKPT)

print(f"\nTokenizer  : {type(tok).__name__} | vocab = {len(tok):,}")
print(f"Model cfg  : vocab_size = {cfg.vocab_size:,}")
print("✓ KHỚP" if len(tok) == cfg.vocab_size
      else f"⚠️ LỆCH {abs(len(tok)-cfg.vocab_size)} token — kiểm tra lại pruning!")

# ---- Sanity: encode thử, id phải nằm trong vocab đã prune ----
probe = tok(dense_pool[0]["question"], dense_pool[0]["candidate_texts"][0],
            truncation=True, max_length=512)
mx = max(probe["input_ids"])
print(f"Max token id: {mx} / {cfg.vocab_size}  {'✓' if mx < cfg.vocab_size else '✗ TRÀN'}")

Files trong checkpoint:
  config.json                          0.00 MB
  model.safetensors                  135.79 MB
  tokenizer.json                       2.50 MB
  tokenizer_config.json                0.00 MB

Tokenizer  : XLMRobertaTokenizerFast | vocab = 36,329
Model cfg  : vocab_size = 36,329
✓ KHỚP
Max token id: 35604 / 36329  ✓


In [10]:
# ===== EXPORT + QUANTIZE =====
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

print("Exporting ONNX...")
ort_fp32 = ORTModelForSequenceClassification.from_pretrained(BEST_CKPT, export=True)
ort_fp32.save_pretrained(ONNX_FP32)
tok.save_pretrained(ONNX_FP32)

print("Quantizing INT8...")
quantizer = ORTQuantizer.from_pretrained(ONNX_FP32)
qconfig   = AutoQuantizationConfig.avx2(is_static=False, per_channel=True)
quantizer.quantize(save_dir=ONNX_INT8, quantization_config=qconfig)
tok.save_pretrained(ONNX_INT8)

mb = lambda p: sum(f.stat().st_size for f in Path(p).rglob("*.onnx")) / 1024**2
print(f"\nPyTorch  : 135.8 MB")
print(f"ONNX fp32: {mb(ONNX_FP32):6.1f} MB")
print(f"ONNX int8: {mb(ONNX_INT8):6.1f} MB  (giảm {(1-mb(ONNX_INT8)/mb(ONNX_FP32))*100:.0f}%)")

del ort_fp32; gc.collect()

Exporting ONNX...
Quantizing INT8...

PyTorch  : 135.8 MB
ONNX fp32:  136.0 MB
ONNX int8:   34.8 MB  (giảm 74%)


3938

In [11]:
# ===== VERIFY INT8 =====
import torch
from tqdm.auto import tqdm
from optimum.onnxruntime import ORTModelForSequenceClassification

def eval_onnx(path, name):
    m = ORTModelForSequenceClassification.from_pretrained(path)
    metrics = []
    for rec in tqdm(dense_pool, desc=name, leave=False):
        docs = rec["candidate_texts"]
        enc = tok([rec["question"]] * len(docs), docs, padding=True,
                  truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            s = m(**enc).logits.squeeze(-1).float().numpy()
        ranked = [rec["candidate_ids"][i] for i in np.argsort(-s)]
        metrics.append(_metrics_one(ranked, rec["gold"]))
    r = summarize_metrics(metrics, name, len(metrics))
    print(f"{name:<11} Hit@1 {r['Hit@1']:.4f} | R@5 {r['Recall@5']:.4f} | "
          f"MRR@10 {r['MRR@10']:.4f} | NDCG@10 {r['NDCG@10']:.4f}")
    del m; gc.collect()
    return r

print("Gốc PyTorch: Hit@1 0.8282 | R@5 0.9590 | MRR@10 0.8844 | NDCG@10 0.9051\n")
r32 = eval_onnx(ONNX_FP32, "ONNX fp32")
r8  = eval_onnx(ONNX_INT8, "ONNX INT8")

d = r8["MRR@10"] - 0.8844
print(f"\n>>> ΔMRR (INT8 vs gốc) = {d:+.4f}  (~{abs(d)*len(dense_pool):.1f} query)")
print(">>> " + ("✓ DÙNG ĐƯỢC" if abs(d) < 0.01 else "⚠️ Mất nhiều — cân nhắc static quant"))

Gốc PyTorch: Hit@1 0.8282 | R@5 0.9590 | MRR@10 0.8844 | NDCG@10 0.9051



ONNX fp32:   0%|          | 0/390 [00:00<?, ?it/s]

Could not find any ONNX files with standard file name model.onnx, files found: [PosixPath('model_quantized.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


ONNX fp32   Hit@1 0.8308 | R@5 0.9590 | MRR@10 0.8856 | NDCG@10 0.9061


ONNX INT8:   0%|          | 0/390 [00:00<?, ?it/s]

ONNX INT8   Hit@1 0.8231 | R@5 0.9641 | MRR@10 0.8834 | NDCG@10 0.9041

>>> ΔMRR (INT8 vs gốc) = -0.0010  (~0.4 query)
>>> ✓ DÙNG ĐƯỢC


In [12]:
# ===== KIỂM TRA + LƯU MODEL VỀ DRIVE =====
import os, shutil
from pathlib import Path

DEST = "/content/drive/MyDrive/Data/archive/deploy"
os.makedirs(DEST, exist_ok=True)

# --- Xem đang có gì ---
for d in [ONNX_FP32, ONNX_INT8]:
    print(f"\n{d}")
    for f in sorted(Path(d).iterdir()):
        print(f"   {f.name:<34} {f.stat().st_size/1024**2:8.2f} MB")

# --- Copy nguyên thư mục sang Drive (dùng được luôn, khỏi giải nén) ---
for src, name in [(ONNX_INT8, "reranker_onnx_int8"), (ONNX_FP32, "reranker_onnx_fp32")]:
    dst = f"{DEST}/{name}"
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"\n✓ Copy → {dst}")

# --- Zip để tải về máy nhanh ---
shutil.make_archive(f"{DEST}/reranker_onnx_int8", "zip", ONNX_INT8)
shutil.make_archive(f"{DEST}/reranker_onnx_fp32", "zip", ONNX_FP32)

print("\n" + "=" * 60)
print("ĐÃ LƯU VÀO DRIVE:")
print("=" * 60)
for f in sorted(Path(DEST).iterdir()):
    size = (f.stat().st_size / 1024**2 if f.is_file()
            else sum(x.stat().st_size for x in f.rglob("*")) / 1024**2)
    print(f"  {f.name:<30} {size:8.1f} MB")
print(f"\n→ Tải zip từ: {DEST}")


/content/onnx_fp32
   config.json                            0.00 MB
   model.onnx                           135.98 MB
   special_tokens_map.json                0.00 MB
   tokenizer.json                         2.50 MB
   tokenizer_config.json                  0.00 MB

/content/onnx_int8
   config.json                            0.00 MB
   model_quantized.onnx                  34.79 MB
   ort_config.json                        0.00 MB
   special_tokens_map.json                0.00 MB
   tokenizer.json                         2.50 MB
   tokenizer_config.json                  0.00 MB

✓ Copy → /content/drive/MyDrive/Data/archive/deploy/reranker_onnx_int8

✓ Copy → /content/drive/MyDrive/Data/archive/deploy/reranker_onnx_fp32

ĐÃ LƯU VÀO DRIVE:
  reranker_onnx_fp32                138.5 MB
  reranker_onnx_fp32.zip            124.7 MB
  reranker_onnx_int8                 37.3 MB
  reranker_onnx_int8.zip             27.2 MB

→ Tải zip từ: /content/drive/MyDrive/Data/archive/deploy


In [13]:
from pathlib import Path
import shutil

D = Path("/content/drive/MyDrive/Data/archive/deploy/reranker_onnx_int8")
q = D / "model_quantized.onnx"
if q.exists():
    q.rename(D / "model.onnx")
    print("✓ Đổi tên → model.onnx")

# zip lại
shutil.make_archive(str(D), "zip", str(D))
print("✓ Zip lại xong")

✓ Đổi tên → model.onnx
✓ Zip lại xong
